In [ ]:
import pandas as pd
import numpy as np
import os
# pip install -U kaleido

In [ ]:

########### Data Folder path ###########
MFI_folder = 'MFI Data'
vr_folder = 'Value Research Data'
MFI_path = os.path.join(MFI_folder)
vr_path = os.path.join(vr_folder)

nav_files = [i for i in os.listdir(MFI_path) if 'Nav' in i or 'NAV' in i or 'nav' in i]
aum_files = [i for i in os.listdir(MFI_path) if 'AUM' in i or 'AUM' in i or 'aum' in i]
map_files = [i for i in os.listdir(MFI_path) if 'MAP' in i or 'Map' in i or 'map' in i]

nav_file_paths = [os.path.join(MFI_folder,i) for i in nav_files]
aum_file_paths = [os.path.join(MFI_folder,i) for i in aum_files]
map_file_paths = [os.path.join(MFI_folder,i) for i in map_files]


In [22]:
"""
load_mfi_data.py
-----------------
Reads raw MFI NAV and AUM files and returns three ready-to-use objects:

    data         : pd.DataFrame  dates × scheme names  (NAV, 0→NaN, ffilled)
    aum          : pd.DataFrame  scheme names × dates  (Adjusted/Segregated removed)
    category_map : pd.DataFrame  scheme metadata from Map.xlsx

Usage:
    from load_mfi_data import data, aum, category_map
"""

import os
import pandas as pd
import numpy as np

MFI_FOLDER = "MFI Data"
ENGINE = "calamine"   # fast Rust reader; falls back to openpyxl if not installed


# ─────────────────────────────────────────────────────────────
# Readers
# ─────────────────────────────────────────────────────────────
def _read_nav(path: str) -> pd.DataFrame:
    """
    NAV xlsx layout:
      rows 0-4  : junk header                       → skiprows=5
      row  5    : Date | scheme_1 | scheme_2 | ...  → header
      rows 6-8  : Scheme/Index Code, AMFI Code, Fund Name
      rows 9+   : MM/DD/YYYY | nav | nav | ...
      last 3    : footer
    """
    df = pd.read_excel(path, skiprows=5, header=0, index_col=0, engine=ENGINE)
    df = df.iloc[3:-3]   # drop 3 metadata rows + 3 footer rows
    df.index = pd.to_datetime(df.index, format="%m/%d/%Y", errors="coerce")
    df = df.loc[df.index.notna()]
    df.index.name = "Date"
    return df.apply(pd.to_numeric, errors="coerce")   # '--' and blanks → NaN


def _read_aum(path: str) -> pd.DataFrame:
    """
    AUM xlsx layout:
      rows 0-4 : junk header                                  → skiprows=5
      row  5   : Fund Name | Scheme Name | Date | Value       → header
      rows 6+  : data
      last 3   : footer
    """
    df = pd.read_excel(path, skiprows=5, header=0, engine=ENGINE)
    df = df.iloc[:-3][["Fund Name", "Scheme Name", "Date", "Value"]].copy()
    df["Date"]  = pd.to_datetime(df["Date"], errors="coerce")
    df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
    return df.dropna(subset=["Date"])


# ─────────────────────────────────────────────────────────────
# Loader
# ─────────────────────────────────────────────────────────────
def load_data(mfi_folder: str = MFI_FOLDER, verbose: bool = True):
    log = print if verbose else (lambda *_: None)

    files = [f for f in os.listdir(mfi_folder)
             if f.endswith(".xlsx") and not f.startswith("~")]
    nav_paths = sorted(os.path.join(mfi_folder, f) for f in files if "nav" in f.lower())
    aum_paths = sorted(os.path.join(mfi_folder, f) for f in files if "aum" in f.lower())

    log(f"NAV files: {[os.path.basename(p) for p in nav_paths]}")
    log(f"AUM files: {[os.path.basename(p) for p in aum_paths]}")

    # ── NAV ───────────────────────────────────────────────────
    log("\nReading NAV files...")
    nav_dfs = []
    for p in nav_paths:
        df = _read_nav(p)
        log(f"  {os.path.basename(p)}: {df.shape[1]} schemes, {len(df)} dates")
        nav_dfs.append(df)
    if not nav_dfs:
        raise RuntimeError("No NAV data loaded.")

    data = pd.concat(nav_dfs, axis=1).replace(0, np.nan).sort_index().ffill()
    log(f"Combined NAV: {data.shape[1]} schemes × {len(data)} dates")

    # ── AUM ───────────────────────────────────────────────────
    log("\nReading AUM files...")
    aum_dfs = []
    for p in aum_paths:
        df = _read_aum(p)
        log(f"  {os.path.basename(p)}: {df['Scheme Name'].nunique()} schemes, {len(df)} rows")
        aum_dfs.append(df)

    aum_long = pd.concat(aum_dfs).drop_duplicates()
    aum = aum_long.pivot_table(values="Value", index="Scheme Name", columns="Date")
    aum = aum.drop([i for i in aum.index if "Adjusted" in i or "Segregated" in i])
    log(f"Combined AUM: {len(aum)} schemes × {aum.shape[1]} months")

    # ── Map ───────────────────────────────────────────────────
    category_map = pd.read_excel(os.path.join(mfi_folder, "Map.xlsx"), engine=ENGINE)
    log(f"Map: {len(category_map)} schemes")

    return data, aum, category_map


# Auto-load on import
if __name__ != "__main__":
    data, aum, category_map = load_data()


# CLI test
if __name__ == "__main__":
    import time
    t0 = time.time()
    data, aum, category_map = load_data(verbose=True)
    print("\n── Sanity checks ──")
    print(f"NAV shape         : {data.shape}")
    print(f"AUM shape         : {aum.shape}")
    print(f"Category map rows : {len(category_map)}")
    print(f"NAV date range    : {data.index.min().date()} → {data.index.max().date()}")
    print(f"AUM date range    : {aum.columns.min().date()} → {aum.columns.max().date()}")
    print(f"\nTotal time: {time.time() - t0:.1f}s")

NAV files: ['HistoricalNav_Report Equity No Flexi.xlsx', 'HistoricalNav_Report Flexi + Rest.xlsx']
AUM files: ['Scheme wise AUM Report-06Jun2026 Equity No Flexi.xlsx', 'Scheme wise AUM Report-06Jun2026 Flexi + Rest.xlsx']

Reading NAV files...
  HistoricalNav_Report Equity No Flexi.xlsx: 490 schemes, 5030 dates
  HistoricalNav_Report Flexi + Rest.xlsx: 416 schemes, 4205 dates
Combined NAV: 906 schemes × 5052 dates

Reading AUM files...
  Scheme wise AUM Report-06Jun2026 Equity No Flexi.xlsx: 490 schemes, 36863 rows
  Scheme wise AUM Report-06Jun2026 Flexi + Rest.xlsx: 436 schemes, 33007 rows
Combined AUM: 916 schemes × 120 months
Map: 908 schemes

── Sanity checks ──
NAV shape         : (5052, 906)
AUM shape         : (916, 120)
Category map rows : 908
NAV date range    : 2006-03-13 → 2026-06-05
AUM date range    : 2016-06-01 → 2026-05-01

Total time: 5.4s


##### Category Mapping - All Fund Houses - From MFI

In [36]:
############ Mapping ###########################
category_map.index = category_map.pop('Scheme Name')
category_map = category_map[['Scheme Sub Nature', 'Category']].copy()   # ← .copy()

# Merge Value + Contra
category_map.loc[
    category_map['Category'].isin(['Contra Fund', 'Value Fund']),
    'Category'
] = 'Value/Contra'

categories = list(set(category_map['Category']))

# Sub-categorise sectoral and thematic funds
thematic_categories = list(set(category_map.loc[category_map['Scheme Sub Nature'] == 'Thematic', 'Category']))
sectoral_categories = list(set(category_map.loc[category_map['Scheme Sub Nature'] == 'Sectoral', 'Category']))
thematic_categories = [i for i in thematic_categories if i not in sectoral_categories]

# Sort so sectoral and thematic sit together at the end
categories = ([i for i in categories if i not in set(thematic_categories) | set(sectoral_categories)]
              + sectoral_categories + thematic_categories)

# Fund house = first word of scheme name (vectorized)
category_map['fund house'] = category_map.index.to_series().str.split(" ").str[0]

# New launches: in NAV but missing from category_map OR from AUM
new_launches = list(
    set(data.columns) - set(category_map.index) | set(data.columns) - set(aum.index)
)
missing = [i for i in data.columns if i not in aum.index and i not in new_launches]

# Keep only schemes present in all three
data = data[aum.index.intersection(category_map.index)]
category_map = category_map.loc[data.columns]

In [16]:
# ############ Mapping ###########################
# category_map  = pd.read_excel('Equity Hybrid Data March 10, 2026.xlsx',sheet_name = "Map")
# # category_map  = pd.read_excel(map_file_paths[0])

# # category_map.index = category_map.pop('Scheme Name')
# # category_cols = [i.split(" -")[0].strip() for i in category_map.index]
# # category_map['clean name']  = category_cols

# category_map.index = category_map.pop('Scheme Name')
# category_map = category_map[['Scheme Sub Nature','Category']]

# # merging value and contra into one
# category_map.loc[(category_map['Category']=='Contra Fund')|(category_map['Category']=='Value Fund'),'Category'] = 'Value/Contra'

# categories = list(set(category_map['Category']))

# # # Sub categorising sectoral and thematic funds
# thematic_categories = list(set(category_map.loc[category_map['Scheme Sub Nature'] == 'Thematic','Category']))
# sectoral_categories = list(set(category_map.loc[category_map['Scheme Sub Nature'] == 'Sectoral','Category']))
# thematic_categories = [i for i in thematic_categories if i not in sectoral_categories]

# # Sorting categories so that sectoral and thematic go hand in hand
# cat = [i for i in categories if i not in set(thematic_categories).union(sectoral_categories)] + sectoral_categories+ thematic_categories
# categories = cat.copy()

# for i in category_map.index : 
#     category_map.loc[i,'fund house'] = i.split(" ")[0]

# ## For these productes either NAV available but AUM not, also if data is available but category map is not available- can be new launces
# new_launches = list(set([i for i in data.columns if i not in category_map.index]).union([i for i in data.columns if i not in aum.index]))
# missing = [i for i in data.columns if i not in aum.index and i not in new_launches]

# ## NAV data and AUM for schemes need to be available : if AUM not avail maybe due to recent launch - check
# ## Making NAV and AUM having same number of schemes
# # data = data[aum.index]
# data = data[aum.index.intersection(category_map.index)]
# # making category map same as nav of schemes, so when picking funds in that scheme doesn't throw error
# category_map = category_map.loc[data.columns]

In [17]:
bottom_up = ['Large & Mid Cap Fund','Small cap Fund','Mid Cap Fund','Multi Cap Fund']
asset_allocation = ['Aggressive Hybrid Fund','Balanced Hybrid Fund','Dynamic Asset Allocation or Balanced Advantage','Conservative Hybrid Fund','Multi Asset Allocation']
top_down = ['Large Cap Fund','Flexi Cap Fund','Focused Fund','ELSS']
arb_plus = ['Equity Savings','Arbitrage Fund']
thematic = [i for i in list(set(category_map['Scheme Sub Nature'])) if i not in set(bottom_up).union(top_down).union(asset_allocation).union(arb_plus)]
category_map.loc[category_map['Scheme Sub Nature'].isin(bottom_up),'Breakdown'] = 'Bottoms Up'
category_map.loc[category_map['Scheme Sub Nature'].isin(top_down),'Breakdown'] = 'Top Down'
category_map.loc[category_map['Scheme Sub Nature'].isin(asset_allocation),'Breakdown'] = 'Asset Allocation'
category_map.loc[category_map['Scheme Sub Nature'].isin(arb_plus),'Breakdown'] = 'Arbitrage+'
category_map.loc[category_map['Scheme Sub Nature'].isin(thematic),'Breakdown'] = 'Thematic'

### Dropping 'WhiteOak','Edelweiss','Mahindra' from Multi Asset Allocation since they have debt taxations
drop_scheme_multi_asset = category_map.loc[(category_map['Scheme Sub Nature'] == 'Multi Asset Allocation') & (category_map['fund house'].isin(['WhiteOak','Edelweiss','Mahindra']))].index
category_map = category_map.drop(drop_scheme_multi_asset)

In [18]:
# Minimum 5 funds need to be there in category for quintile computation
category_count = (category_map.reset_index()).groupby('Category').count()
# min_funds_category  = list(category_count.loc[category_count['Scheme Name']>=4].index)
min_funds_category  = list(category_count.loc[category_count['fund house']>=4].index)
categories = [i for i in categories if i in min_funds_category]

In [ ]:
#########################################################################################################################################

### Category Mapping - Exact peer based - From Value Research

In [ ]:
data1 = data.copy()
# Get scheme names whether data1.columns is flat or already a MultiIndex
schemes = (data1.columns.get_level_values(-1) if isinstance(data1.columns, pd.MultiIndex) else data1.columns)
# Rebuild MultiIndex from scratch
data1.columns = pd.MultiIndex.from_tuples(list(zip(category_map.loc[schemes, 'Category'].values, schemes)))
# Sort: by category, then Aditya Birla first, then alphabetical
data1 = data1[sorted(data1.columns,key=lambda c: (str(c[0]), 0 if c[1].startswith("Aditya Birla") else 1, c[1]))]

In [79]:
'Flexi Cap Fund','Large Cap Fund','Large & Mid Cap Fund'

('Flexi Cap Fund', 'Large Cap Fund', 'Large & Mid Cap Fund')

In [83]:
data1['Large & Mid Cap Fund'].to_clipboard()

In [78]:
category_map.to_clipboard()

In [63]:
data1.to_clipboard()

In [29]:
vr_map_file = "Value Research Benchmarks NAV March 10, 2026.xlsx"
category_map_exact = pd.read_excel(os.path.join(vr_folder,vr_map_file),sheet_name = "Ret. Compr.(Equity) - Dir")
category_map_exact.index = category_map_exact.pop('Scheme')
####### Finding Schemes which are in Value Research Peerset File but not in MFI NAV data
missing_peers_nav = list(category_map_exact.loc[pd.isna(category_map_exact['Scheme Name From MFI']) & (category_map_exact['AMFI Code'] != 'bench')].index)
print("' below schemes are in Value Research Peerset but No NAV Data From MFI Dump\n",missing_peers_nav,"\n dropping these schemes")
category_map_exact = category_map_exact.drop(missing_peers_nav)
for i in category_map_exact.index : 
    category_map_exact.loc[i,'fund house'] = i.split(" ")[0]

#### Benchmarks from exact peer category of value research
exact_bench = category_map_exact.loc[category_map_exact['AMFI Code'] == 'bench',['Category']]
exact_bench = exact_bench.reset_index().set_index('Category')
exact_bench.columns = ['Benchmark']
#### Dropping Benchmarks from exact peer category mapping of value research
category_map_exact = category_map_exact.drop(exact_bench['Benchmark'])
category_map_exact = category_map_exact.reset_index().set_index('Scheme Name From MFI')

######## Selecting only Value Research peerset data #################
# vr_peer_schemes = list(set(category_map_exact['Scheme Name From MFI']))
# vr_peer_schemes = [i for i in vr_peer_schemes if pd.isna(i)==False]
# aum = aum.loc[vr_peer_schemes]
# data = data[vr_peer_schemes]

' below schemes are in Value Research Peerset but No NAV Data From MFI Dump
 ['Aditya Birla Sun Life US Equity Passive FoF - Direct Plan', 'Axis US Specific Equity Passive FoF - Direct Plan', 'Motilal Oswal Nasdaq 100 FOF - Direct Plan'] 
 dropping these schemes


#### Benchmarks - For Scoring on exact peer set from Value Research

In [ ]:
from functools import reduce
bench_nav_bbg = pd.read_excel('Value Research Benchmarks NAV March 10, 2026.xlsx',sheet_name = 'Bloomberg Bench NAV')
bench_nav_crisil = pd.read_excel('Value Research Benchmarks NAV March 10, 2026.xlsx',sheet_name = 'Crisil Bench NAV')
bench_nav_vr = pd.read_excel('Value Research Benchmarks NAV March 10, 2026.xlsx',sheet_name = 'VR Bench NAV')
bench_nav_other = pd.read_excel('Value Research Benchmarks NAV March 10, 2026.xlsx',sheet_name = 'Silver')
bench_nav_bbg.set_index('Dates',inplace=True)
bench_nav_crisil.set_index('Date',inplace=True)
bench_nav_vr.set_index('Date',inplace=True)
bench_nav_other.set_index('NAV Date',inplace=True)
bench_nav = reduce(lambda left, right: pd.merge(left, right, left_index=True, right_index=True,how='left'), 
                   [bench_nav_bbg, bench_nav_crisil, bench_nav_vr, bench_nav_other])
bench_name_map = pd.read_excel('Value Research Benchmarks NAV March 10, 2026.xlsx',sheet_name = 'Benchmarks and sources')
bench_name_map.set_index('Identifier',inplace=True)
bench_nav.columns = [bench_name_map.loc[i,'Benchmarks'] for i in bench_nav.columns]

In [ ]:
common_dates = data.index.intersection(bench_nav.index)
bench_nav = bench_nav.loc[common_dates]
data = data.loc[common_dates]
data.drop_duplicates(inplace=True)
bench_nav.drop_duplicates(inplace=True)

## Calculations

In [ ]:
# all_houses = list(set(category_map_exact.loc[category_map_exact['AMFI Code'] != 'bench','fund house']))

In [ ]:
top15 = ['Axis','Franklin','Kotak','HDFC','Aditya','DSP','UTI','Invesco','Canara','SBI','Mirae','Nippon','Tata','HSBC','ICICI']
all_houses = list(set(category_map['fund house']))
all_houses.sort()

In [ ]:
#  ffill in AUM since not all schemes have reported AUM historically on same periodicity
aum1 = aum.copy()
aum1 = aum1.T
aum1 = aum1.sort_index()
aum1 = aum1.replace('--',np.nan)
aum1 = aum1.ffill()
aum1 = aum1.T
aum2 = aum1.iloc[:,1:].T

for i in aum1.index : 
    aum1.loc[i,'fund house'] = i.split()[0]
aum1.insert(0,'fund house' ,aum1.pop('fund house')) 
fund_aum = aum1.groupby('fund house').sum()
funds = list(fund_aum.index)


percent_aum_df = pd.DataFrame()
for fund in all_houses :
    percent_aum = aum1.loc[aum1['fund house'].isin([fund])].iloc[:,1:]/fund_aum.loc[fund]
    percent_aum = percent_aum.T
    
    # making last availabale %of AUM (monthly AUM) asa daily % of AUM 
    percent_aum_ = pd.DataFrame([1]*len(data) ,index = data.index)
    percent_aum_ = percent_aum_.merge(percent_aum,left_index=True,right_index=True,how='outer').ffill().drop(0,axis=1)
    percent_aum_ = percent_aum_.loc[data.index]
    percent_aum_df = percent_aum_df.merge(percent_aum_,left_index=True,right_index=True,how='outer')
    
percent_aum_df1 = category_map[['fund house']].merge(percent_aum_df.T,left_index=True,right_index=True,how='right')

## Optional - to see how much total AUM exposure of each fund house is in our peerset 
##########################################
exact_peer_based_aum_fundwise = category_map_exact.merge(percent_aum_df1,left_index=True,right_index=True).groupby('fund house_x').sum().drop(['Scheme','fund house_y','AMFI Code','Category'],axis=1)

vr_scheme_count = pd.DataFrame(category_map_exact.groupby(['Scheme Name From MFI']).count()['Scheme'].sort_values(ascending=False))
vr_scheme_count['Fund House'] = [i[0] for i in vr_scheme_count.index.str.split(' ')]
vr_repeated_scheme = vr_scheme_count.loc[vr_scheme_count['Scheme']==2].groupby('Fund House').count()
# exact_peer_based_aum_fundwise.to_clipboard()
##########################################

In [ ]:
# exact_peer_based_aum_fundwise = category_map_exact.loc[category_map_exact['Category'].isin(categories_exact)].merge(percent_aum_df1,left_index=True,right_index=True).groupby('fund house_x').sum().drop(['Scheme','fund house_y','AMFI Code','Category'],axis=1)

# vr_scheme_count = pd.DataFrame(category_map_exact.groupby(['Scheme Name From MFI']).count()['Scheme'].sort_values(ascending=False))
# vr_scheme_count['Fund House'] = [i[0] for i in vr_scheme_count.index.str.split(' ')]
# vr_repeated_scheme = vr_scheme_count.loc[vr_scheme_count['Scheme']==2].groupby('Fund House').count()

### Sleeve Analysis

In [ ]:
################################### Main Script - All Schemes All Fund Houses - Used in Vantage Point ############################################################################
#### Final Variation which is in current use - 1 year and 3 years - Parallel - Faster Version
############## Top - N Schemes in Category / All Schemes in Category ####################################
from joblib import Parallel, delayed
import pandas as pd
categories = list(set(category_map['Category']))


####################################################################################################
_3y_rets = (data/data.shift(250*3))**(1/3)-1
_1y_rets = data/data.shift(250)-1
# Global DataFrames (same as before)
qy_df_1y = pd.DataFrame(index=data.index, columns=data.columns)
qy_df_3y = pd.DataFrame(index=data.index, columns=data.columns)
distance_df_1y = pd.DataFrame()
distance_df_3y = pd.DataFrame()
percentile_rolling_df_1y = pd.DataFrame()
percentile_rolling_df_3y = pd.DataFrame()

####################################################################################################
# Wrap the code block for one category into a function.
def process_category(f):
    # Get the funds for this category.
#     fs = list(category_map[category_map['Scheme Sub Nature'] == f].index)
    # For all categories for all funds (uncomment alternative definitions if needed)
    fs = list(category_map[category_map['Category'] == f].index)
    # For top15 fund houses (if needed):
    # fs = list(category_map.loc[(category_map['Category']==f) & (category_map['fund house'].isin(top15))].index)
    
    ####################### Rolling returns #######################
    # 1Y and 3y returns for the funds in this category
    f_1y_rets = _1y_rets[fs]
    f_3y_rets = _3y_rets[fs]
    
    # Get only the dates where at least one fund has a return value.
    date_list = f_3y_rets.loc[pd.isna(f_3y_rets).sum(axis=1) != len(f_3y_rets.columns)].index
    date_list = [i for i in date_list if i > min(aum2.index)]

    # Create local DataFrames for quartile assignments and distances
    local_qy_df_1y = pd.DataFrame(index=f_1y_rets.index, columns=fs)
    local_qy_df_3y = pd.DataFrame(index=f_3y_rets.index, columns=fs)
    local_distance_df_1y = pd.DataFrame()
    local_distance_df_3y = pd.DataFrame()

    for d in date_list:
        n = 20
        ######### 1-year rets quartile 
        f_1y_rets_available = f_1y_rets.loc[d][pd.isna(f_1y_rets.loc[d]) == False]
        f_1y_rets_available_sorted = f_1y_rets_available.sort_values(ascending=False)
        
        ############### Selecting how many top schemes to select for the category #####################
#         topn_schemes = (aum1.loc[:,[i for i in aum1.columns[1:] if i < d][-1]][f_1y_rets_available.index].sort_values(ascending=False)[:n].index)
        topn_schemes = f_1y_rets_available.index
        
        f_1y_rets_available = f_1y_rets_available[[i for i in f_1y_rets_available.index if i in topn_schemes]]
        f_1y_rets_available_sorted = f_1y_rets_available_sorted[[i for i in f_1y_rets_available_sorted.index if i in topn_schemes]]
        
        ## Rounding up when number of funds is not divisible by 4
        items_each_bucket = len(f_1y_rets_available) // 4
        extra_items = len(f_1y_rets_available) % 4
        bucket_items = [items_each_bucket] * 4
        for i in range(1, extra_items + 1):
            bucket_items[i - 1] = bucket_items[i] + 1

        q1_funds_1y = list(f_1y_rets_available_sorted[:bucket_items[0]].index)
        q2_funds_1y = list(f_1y_rets_available_sorted[bucket_items[0]:][:bucket_items[1]].index)
        q3_funds_1y = list(f_1y_rets_available_sorted[bucket_items[0] + bucket_items[1]:][:bucket_items[2]].index)
        q4_funds_1y = list(f_1y_rets_available_sorted[bucket_items[0] + bucket_items[1] + bucket_items[2]:][:bucket_items[3]].index)

        ######### 3 years rets quartile  
        f_3y_rets_available = f_3y_rets.loc[d][pd.isna(f_3y_rets.loc[d]) == False]
        f_3y_rets_available_sorted = f_3y_rets_available.sort_values(ascending=False)
        
        ############### Selecting how many top schemes to select for the category #####################
#         topn_schemes = (aum1.loc[:,[i for i in aum1.columns[1:] if i < d][-1]][f_3y_rets_available.index].sort_values(ascending=False)[:n].index)
        topn_schemes = f_3y_rets_available.index
        
        f_3y_rets_available = f_3y_rets_available[[i for i in f_3y_rets_available.index if i in topn_schemes]]
        f_3y_rets_available_sorted = f_3y_rets_available_sorted[[i for i in f_3y_rets_available_sorted.index if i in topn_schemes]] 

        ## Rounding up when number of funds is not divisible by 4
        items_each_bucket = len(f_3y_rets_available) // 4
        extra_items = len(f_3y_rets_available) % 4
        bucket_items = [items_each_bucket] * 4
        for i in range(1, extra_items + 1):
            bucket_items[i - 1] = bucket_items[i] + 1

        q1_funds_3y = list(f_3y_rets_available_sorted[:bucket_items[0]].index)
        q2_funds_3y = list(f_3y_rets_available_sorted[bucket_items[0]:][:bucket_items[1]].index)
        q3_funds_3y = list(f_3y_rets_available_sorted[bucket_items[0] + bucket_items[1]:][:bucket_items[2]].index)
        q4_funds_3y = list(f_3y_rets_available_sorted[bucket_items[0] + bucket_items[1] + bucket_items[2]:][:bucket_items[3]].index)

        ########################## Assigning quartiles  ########################################
        local_qy_df_1y.loc[d, q1_funds_1y] = 'q1'
        local_qy_df_1y.loc[d, q2_funds_1y] = 'q2'
        local_qy_df_1y.loc[d, q3_funds_1y] = 'q3'
        local_qy_df_1y.loc[d, q4_funds_1y] = 'q4'

        local_qy_df_3y.loc[d, q1_funds_3y] = 'q1'
        local_qy_df_3y.loc[d, q2_funds_3y] = 'q2'
        local_qy_df_3y.loc[d, q3_funds_3y] = 'q3'
        local_qy_df_3y.loc[d, q4_funds_3y] = 'q4'
        
        ######################### Distance in returns from top and bottom performing funds ############################
        ############################### Return of Top, Bottom, Mean etc #########################################################
        ###################### 1 year
        aditya_fund_in_cat_1y = [i for i in f_1y_rets_available_sorted.index if 'Aditya' in i]
        if len(aditya_fund_in_cat_1y) > 1:
            print('More than one ABSL funds in cat ', f)

        local_distance_df_1y.loc[d, 'Category'] = f
        local_distance_df_1y.loc[d, 'category top return'] = f_1y_rets_available_sorted[0]
        local_distance_df_1y.loc[d, 'category bottom return'] = f_1y_rets_available_sorted[-1]
        if len(f_1y_rets_available_sorted) > 2:
#             local_distance_df_1y.loc[d, 'category mean return'] = f_1y_rets_available_sorted[int(len(f_1y_rets_available_sorted)/2) + (len(f_1y_rets_available_sorted) % 2 > 0)]
            local_distance_df_1y.loc[d, 'category mean return'] = f_1y_rets_available_sorted[int(len(f_1y_rets_available_sorted)/2)]
        if len(f_1y_rets_available_sorted) > 3:
            local_distance_df_1y.loc[d, 'category Q1 return'] = f_1y_rets_available_sorted[q1_funds_1y][-1]
        if len(aditya_fund_in_cat_1y) > 0:
            aditya_ret_1y = f_1y_rets_available_sorted[aditya_fund_in_cat_1y].values[0]
            local_distance_df_1y.loc[d, 'category Birla scheme return'] = aditya_ret_1y
            
#             behind_birla = f_1y_rets_available_sorted.loc[(aditya_ret_1y > f_1y_rets_available_sorted)].index
            behind_birla = f_1y_rets_available_sorted.loc[(aditya_ret_1y >= f_1y_rets_available_sorted)].index
            local_distance_df_1y.loc[d,'% of AUM in category Birla is outperforming'] = aum2.loc[:d,behind_birla].iloc[-1].sum()/aum2.loc[:d,f_1y_rets_available_sorted.index].iloc[-1].sum()
            local_distance_df_1y.loc[d,'% of funds in category Birla is outperforming'] = len(behind_birla)/len(f_1y_rets_available_sorted)

        local_distance_df_1y.loc[d, 'category top scheme'] = f_1y_rets_available_sorted.index[0]
        local_distance_df_1y.loc[d, 'category bottom scheme'] = f_1y_rets_available_sorted.index[-1]
        if len(f_1y_rets_available_sorted) > 2:
            local_distance_df_1y.loc[d, 'category mean scheme'] = f_1y_rets_available_sorted.index[int(len(f_1y_rets_available_sorted)/2) + (len(f_1y_rets_available_sorted) % 2 > 0)]
        local_distance_df_1y.loc[d, 'category Q1 scheme'] = f_1y_rets_available_sorted[q1_funds_1y].index[-1]
        if len(aditya_fund_in_cat_1y) > 0:
            local_distance_df_1y.loc[d, 'category Birla scheme'] = aditya_fund_in_cat_1y[0]
            ###################### 3 year
        aditya_fund_in_cat_3y = [i for i in f_3y_rets_available_sorted.index if 'Aditya' in i]
        if len(aditya_fund_in_cat_3y) > 1:
            print('More than one ABSL funds in cat ', f)

        local_distance_df_3y.loc[d, 'Category'] = f
        local_distance_df_3y.loc[d, 'category top return'] = f_3y_rets_available_sorted[0]
        local_distance_df_3y.loc[d, 'category bottom return'] = f_3y_rets_available_sorted[-1]
        if len(f_3y_rets_available_sorted) > 2:
            local_distance_df_3y.loc[d, 'category mean return'] = f_3y_rets_available_sorted[int(len(f_3y_rets_available_sorted)/2)]
        if len(f_3y_rets_available_sorted) > 3:
            local_distance_df_3y.loc[d, 'category Q1 return'] = f_3y_rets_available_sorted[q1_funds_3y][-1]
        if len(aditya_fund_in_cat_3y) > 0:
            aditya_ret_3y = f_3y_rets_available_sorted[aditya_fund_in_cat_3y].values[0]
            local_distance_df_3y.loc[d, 'category Birla scheme return'] = aditya_ret_3y
            behind_birla = f_3y_rets_available_sorted.loc[(aditya_ret_3y >= f_3y_rets_available_sorted)].index
            local_distance_df_3y.loc[d,'% of AUM in category Birla is outperforming'] = aum2.loc[:d,behind_birla].iloc[-1].sum()/aum2.loc[:d,f_3y_rets_available_sorted.index].iloc[-1].sum()
            local_distance_df_3y.loc[d,'% of funds in category Birla is outperforming'] = len(behind_birla)/len(f_3y_rets_available_sorted)


        local_distance_df_3y.loc[d, 'category top scheme'] = f_3y_rets_available_sorted.index[0]
        local_distance_df_3y.loc[d, 'category bottom scheme'] = f_3y_rets_available_sorted.index[-1]
        if len(f_3y_rets_available_sorted) > 2:
            local_distance_df_3y.loc[d, 'category mean scheme'] = f_3y_rets_available_sorted.index[int(len(f_3y_rets_available_sorted)/2) + (len(f_3y_rets_available_sorted) % 2 > 0)]
        local_distance_df_3y.loc[d, 'category Q1 scheme'] = f_3y_rets_available_sorted[q1_funds_3y].index[-1]
        if len(aditya_fund_in_cat_3y) > 0:
            local_distance_df_3y.loc[d, 'category Birla scheme'] = aditya_fund_in_cat_3y[0]

    ######################  %tile in returns ########################################
    # 1Y
    f_1y_per = f_1y_rets.rank(axis=1).div(len(f_1y_rets.columns) - pd.isna(f_1y_rets).sum(axis=1), axis=0)
    f_1y_per = f_1y_per.loc[f_1y_rets.index]
    # 3y
    f_3y_per = f_3y_rets.rank(axis=1).div(len(f_3y_rets.columns) - pd.isna(f_3y_rets).sum(axis=1), axis=0)
    f_3y_per = f_3y_per.loc[f_3y_rets.index]

    # Return all the local outputs for this category.
    return {
        'f': f,
        'fs': fs,
        'qy_df_1y': local_qy_df_1y,
        'qy_df_3y': local_qy_df_3y,
        'distance_df_1y': local_distance_df_1y,
        'distance_df_3y': local_distance_df_3y,
        'f_1y_per': f_1y_per,
        'f_3y_per': f_3y_per
        }
####################################################################################################
# Run the processing in parallel over all categories.
results = Parallel(n_jobs=-1)(delayed(process_category)(f) for f in categories)
####################################################################################################
# Merge the results from all categories back into the global DataFrames.
for res in results:
    fs = res['fs']
    # Update the quartile assignments (only for the funds in this category)
    qy_df_1y.loc[:, fs] = res['qy_df_1y']
    qy_df_3y.loc[:, fs] = res['qy_df_3y']
    # Concatenate distance DataFrames.
    distance_df_1y = pd.concat([distance_df_1y, res['distance_df_1y']])
    distance_df_3y = pd.concat([distance_df_3y, res['distance_df_3y']])
    # Merge percentile data.
    percentile_rolling_df_1y = percentile_rolling_df_1y.merge(res['f_1y_per'], left_index=True, right_index=True, how='outer')
    percentile_rolling_df_3y = percentile_rolling_df_3y.merge(res['f_3y_per'], left_index=True, right_index=True, how='outer')
####################################################################################################
# (The remaining code below stays the same.)
percent_in_q1_1y = (qy_df_1y.T == 'q1') * percent_aum_df1.iloc[:, 1:]
percent_in_q2_1y = (qy_df_1y.T == 'q2') * percent_aum_df1.iloc[:, 1:]
percent_in_q3_1y = (qy_df_1y.T == 'q3') * percent_aum_df1.iloc[:, 1:]
percent_in_q4_1y = (qy_df_1y.T == 'q4') * percent_aum_df1.iloc[:, 1:]
percent_in_q1_3y = (qy_df_3y.T == 'q1') * percent_aum_df1.iloc[:, 1:]
percent_in_q2_3y = (qy_df_3y.T == 'q2') * percent_aum_df1.iloc[:, 1:]
percent_in_q3_3y = (qy_df_3y.T == 'q3') * percent_aum_df1.iloc[:, 1:]
percent_in_q4_3y = (qy_df_3y.T == 'q4') * percent_aum_df1.iloc[:, 1:]

percent_in_q1_1y['Quntile'] = '% in Q1'
percent_in_q2_1y['Quntile'] = '% in Q2'
percent_in_q3_1y['Quntile'] = '% in Q3'
percent_in_q4_1y['Quntile'] = '% in Q4'
percent_in_q1_3y['Quntile'] = '% in Q1'
percent_in_q2_3y['Quntile'] = '% in Q2'
percent_in_q3_3y['Quntile'] = '% in Q3'
percent_in_q4_3y['Quntile'] = '% in Q4'

percent_in_q_1y = pd.concat([percent_in_q1_1y, percent_in_q2_1y, percent_in_q3_1y, percent_in_q4_1y])
percent_in_q_3y = pd.concat([percent_in_q1_3y, percent_in_q2_3y, percent_in_q3_3y, percent_in_q4_3y])
percent_in_q_1y = percent_in_q_1y.loc[pd.isna(percent_in_q_1y.iloc[:, :-1]).sum(axis=1) != len(percent_in_q_1y.iloc[:, :-1].columns)]
percent_in_q_3y = percent_in_q_3y.loc[pd.isna(percent_in_q_3y.iloc[:, :-1]).sum(axis=1) != len(percent_in_q_3y.iloc[:, :-1].columns)]
#####################################################################################################
merged_1y = percent_in_q_1y.merge(category_map, left_index=True, right_index=True)
merged_3y = percent_in_q_3y.merge(category_map, left_index=True, right_index=True)
merged_1y['Scheme'] = merged_1y.index
merged_3y['Scheme'] = merged_3y.index
merged_1y['Rolling Window'] = '1 Year'
merged_3y['Rolling Window'] = '3 Year'
merged_1y_grouped = merged_1y.groupby(['Rolling Window','Breakdown', 'Category', 'Scheme Sub Nature', 'fund house', 'Scheme', 'Quntile']).sum()
merged_3y_grouped = merged_3y.groupby(['Rolling Window','Breakdown', 'Category', 'Scheme Sub Nature', 'fund house', 'Scheme', 'Quntile']).sum()
merged_quartile_df = pd.concat([merged_3y_grouped,merged_1y_grouped])
truncate_from = pd.to_datetime('2013-12-31')
merged_quartile_df = merged_quartile_df.loc[:,[i for i in merged_quartile_df.columns if i >= truncate_from]]
####################################################################################################
distance_df_1y['Rolling window'] = '1 Year'
distance_df_3y['Rolling window'] = '3 Year'

distance_df_3y['Birla - category Average'] = distance_df_3y['category Birla scheme return'] - distance_df_3y['category mean return']
distance_df_3y['Birla - Category Top'] = distance_df_3y['category Birla scheme return'] - distance_df_3y['category top return']
distance_df_3y['Birla - Category Bottom'] = distance_df_3y['category Birla scheme return'] - distance_df_3y['category bottom return']
distance_df_3y['Birla - Q1'] = distance_df_3y['category Birla scheme return'] - distance_df_3y['category Q1 return']

distance_df_1y['Birla - category Average'] = distance_df_1y['category Birla scheme return'] - distance_df_1y['category mean return']
distance_df_1y['Birla - Category Top'] = distance_df_1y['category Birla scheme return'] - distance_df_1y['category top return']
distance_df_1y['Birla - Category Bottom'] = distance_df_1y['category Birla scheme return'] - distance_df_1y['category bottom return']
distance_df_1y['Birla - Q1'] = distance_df_1y['category Birla scheme return'] - distance_df_1y['category Q1 return']

# for c in categories : 
#     distance_df_1y.loc[distance_df_1y['Category']==c,'% times Birla with +ve alpha over category average'] = (distance_df_1y.loc[distance_df_1y['Category']==c]['Birla - category Average']>0).rolling(window=250).sum()/250
#     distance_df_3y.loc[distance_df_3y['Category']==c,'% times Birla with +ve alpha over category average'] = (distance_df_3y.loc[distance_df_3y['Category']==c]['Birla - category Average']>0).rolling(window=250).sum()/250 
# merged_distance_df = pd.concat([distance_df_3y,distance_df_1y])
# reorder_cols = ['Rolling window','Category','category top scheme','category bottom scheme', 'category mean scheme',
#                 'category Q1 scheme','category Birla scheme', 'category top return', 'category bottom return','category mean return',
#                 'category Q1 return','category Birla scheme return','Birla - category Average','Birla - Category Top',
#                 'Birla - Category Bottom','Birla - Q1','% times Birla with +ve alpha over category average']
# merged_distance_df = merged_distance_df[reorder_cols]
# merged_distance_df = merged_distance_df.sort_values(by = ['Rolling window','Category'])
####################################################################################################
for c in categories : 
    distance_df_1y.loc[distance_df_1y['Category']==c,'% times Birla with +ve alpha over category average'] = (distance_df_1y.loc[distance_df_1y['Category']==c]['Birla - category Average']>0).rolling(window=250).sum()/250
    distance_df_3y.loc[distance_df_3y['Category']==c,'% times Birla with +ve alpha over category average'] = (distance_df_3y.loc[distance_df_3y['Category']==c]['Birla - category Average']>0).rolling(window=250).sum()/250 
merged_distance_df = pd.concat([distance_df_3y,distance_df_1y])
reorder_cols = columns = ["Rolling window", 
    "Category", 
    "category top return", 
    "category bottom return", 
    "category mean return", 
    "category Q1 return", 
    "category Birla scheme return", 
    "% of AUM in category Birla is outperforming", 
    "% of funds in category Birla is outperforming", 
    "Birla - category Average", 
    "Birla - Category Top", 
    "Birla - Category Bottom", 
    "Birla - Q1", 
    "% times Birla with +ve alpha over category average", 
    "category top scheme", 
    "category bottom scheme", 
    "category mean scheme", 
    "category Q1 scheme", 
    "category Birla scheme"]

merged_distance_df = merged_distance_df[reorder_cols]
merged_distance_df = merged_distance_df.sort_values(by = ['Rolling window','Category'])

In [ ]:
################################### Main Script copy - Value research peerset ############################################################################
#### Final Variation which is in current use - 1 year and 3 years - Parallel - Faster Version
############## Top - N Schemes in Category / All Schemes in Category ####################################

from joblib import Parallel, delayed
import pandas as pd
categories = list(set(category_map['Category']))

categories_exact = list(set(category_map_exact['Category']))
avoid_categories = ["Domestic + International",'Multi Index FoF','Other Competitor Thematic/Contra Funds']
categories_exact = [i for i in categories_exact if i not in avoid_categories]
## Sorting so that LargeCap comes after focused category - since both have same peer group
categories_exact.sort()


####################################################################################################
_3y_rets = (data/data.shift(250*3))**(1/3)-1
_1y_rets = data/data.shift(250)-1
# Global DataFrames (same as before)
qy_df_1y = pd.DataFrame(index=data.index, columns=data.columns)
qy_df_3y = pd.DataFrame(index=data.index, columns=data.columns)
distance_df_1y = pd.DataFrame()
distance_df_3y = pd.DataFrame()
percentile_rolling_df_1y = pd.DataFrame()
percentile_rolling_df_3y = pd.DataFrame()

####################################################################################################
# Wrap the code block for one category into a function.
def process_category(f):
    # Get the funds for this category.
#     fs = list(category_map[category_map['Scheme Sub Nature'] == f].index)

    # For all categories for all funds (uncomment alternative definitions if needed)
    # fs = list(category_map[category_map['Category'] == f].index)
    ### Only value research categories
    fs = list(category_map_exact.loc[category_map_exact['Category'] ==f].index)

    # For top15 fund houses (if needed):
    # fs = list(category_map.loc[(category_map['Category']==f) & (category_map['fund house'].isin(top15))].index)
    
    ####################### Rolling returns #######################
    # 1Y and 3y returns for the funds in this category
    f_1y_rets = _1y_rets[fs]
    f_3y_rets = _3y_rets[fs]
    
    # Get only the dates where at least one fund has a return value.
    date_list = f_3y_rets.loc[pd.isna(f_3y_rets).sum(axis=1) != len(f_3y_rets.columns)].index
    date_list = [i for i in date_list if i > min(aum2.index)]

    # Create local DataFrames for quartile assignments and distances
    local_qy_df_1y = pd.DataFrame(index=f_1y_rets.index, columns=fs)
    local_qy_df_3y = pd.DataFrame(index=f_3y_rets.index, columns=fs)
    local_distance_df_1y = pd.DataFrame()
    local_distance_df_3y = pd.DataFrame()

    for d in date_list:
        n = 20
        ######### 1-year rets quartile 
        f_1y_rets_available = f_1y_rets.loc[d][pd.isna(f_1y_rets.loc[d]) == False]
        f_1y_rets_available_sorted = f_1y_rets_available.sort_values(ascending=False)
        
        ############### Selecting how many top schemes to select for the category #####################
#         topn_schemes = (aum1.loc[:,[i for i in aum1.columns[1:] if i < d][-1]][f_1y_rets_available.index].sort_values(ascending=False)[:n].index)
        topn_schemes = f_1y_rets_available.index
        
        f_1y_rets_available = f_1y_rets_available[[i for i in f_1y_rets_available.index if i in topn_schemes]]
        f_1y_rets_available_sorted = f_1y_rets_available_sorted[[i for i in f_1y_rets_available_sorted.index if i in topn_schemes]]
        
        ## Rounding up when number of funds is not divisible by 4
        items_each_bucket = len(f_1y_rets_available) // 4
        extra_items = len(f_1y_rets_available) % 4
        bucket_items = [items_each_bucket] * 4
        for i in range(1, extra_items + 1):
            bucket_items[i - 1] = bucket_items[i] + 1

        q1_funds_1y = list(f_1y_rets_available_sorted[:bucket_items[0]].index)
        q2_funds_1y = list(f_1y_rets_available_sorted[bucket_items[0]:][:bucket_items[1]].index)
        q3_funds_1y = list(f_1y_rets_available_sorted[bucket_items[0] + bucket_items[1]:][:bucket_items[2]].index)
        q4_funds_1y = list(f_1y_rets_available_sorted[bucket_items[0] + bucket_items[1] + bucket_items[2]:][:bucket_items[3]].index)

        ######### 3 years rets quartile  
        f_3y_rets_available = f_3y_rets.loc[d][pd.isna(f_3y_rets.loc[d]) == False]
        f_3y_rets_available_sorted = f_3y_rets_available.sort_values(ascending=False)
        
        ############### Selecting how many top schemes to select for the category #####################
#         topn_schemes = (aum1.loc[:,[i for i in aum1.columns[1:] if i < d][-1]][f_3y_rets_available.index].sort_values(ascending=False)[:n].index)
        topn_schemes = f_3y_rets_available.index
        
        f_3y_rets_available = f_3y_rets_available[[i for i in f_3y_rets_available.index if i in topn_schemes]]
        f_3y_rets_available_sorted = f_3y_rets_available_sorted[[i for i in f_3y_rets_available_sorted.index if i in topn_schemes]] 

        ## Rounding up when number of funds is not divisible by 4
        items_each_bucket = len(f_3y_rets_available) // 4
        extra_items = len(f_3y_rets_available) % 4
        bucket_items = [items_each_bucket] * 4
        for i in range(1, extra_items + 1):
            bucket_items[i - 1] = bucket_items[i] + 1

        q1_funds_3y = list(f_3y_rets_available_sorted[:bucket_items[0]].index)
        q2_funds_3y = list(f_3y_rets_available_sorted[bucket_items[0]:][:bucket_items[1]].index)
        q3_funds_3y = list(f_3y_rets_available_sorted[bucket_items[0] + bucket_items[1]:][:bucket_items[2]].index)
        q4_funds_3y = list(f_3y_rets_available_sorted[bucket_items[0] + bucket_items[1] + bucket_items[2]:][:bucket_items[3]].index)

        ########################## Assigning quartiles  ########################################
        local_qy_df_1y.loc[d, q1_funds_1y] = 'q1'
        local_qy_df_1y.loc[d, q2_funds_1y] = 'q2'
        local_qy_df_1y.loc[d, q3_funds_1y] = 'q3'
        local_qy_df_1y.loc[d, q4_funds_1y] = 'q4'

        local_qy_df_3y.loc[d, q1_funds_3y] = 'q1'
        local_qy_df_3y.loc[d, q2_funds_3y] = 'q2'
        local_qy_df_3y.loc[d, q3_funds_3y] = 'q3'
        local_qy_df_3y.loc[d, q4_funds_3y] = 'q4'
        
        ######################### Distance in returns from top and bottom performing funds ############################
        ############################### Return of Top, Bottom, Mean etc #########################################################
        ###################### 1 year
        aditya_fund_in_cat_1y = [i for i in f_1y_rets_available_sorted.index if 'Aditya' in i]
        if len(aditya_fund_in_cat_1y) > 1:
            print('More than one ABSL funds in cat ', f)

        local_distance_df_1y.loc[d, 'Category'] = f
        local_distance_df_1y.loc[d, 'category top return'] = f_1y_rets_available_sorted[0]
        local_distance_df_1y.loc[d, 'category bottom return'] = f_1y_rets_available_sorted[-1]
        if len(f_1y_rets_available_sorted) > 2:
#             local_distance_df_1y.loc[d, 'category mean return'] = f_1y_rets_available_sorted[int(len(f_1y_rets_available_sorted)/2) + (len(f_1y_rets_available_sorted) % 2 > 0)]
            local_distance_df_1y.loc[d, 'category mean return'] = f_1y_rets_available_sorted[int(len(f_1y_rets_available_sorted)/2)]
        if len(f_1y_rets_available_sorted) > 3:
            local_distance_df_1y.loc[d, 'category Q1 return'] = f_1y_rets_available_sorted[q1_funds_1y][-1]
        if len(aditya_fund_in_cat_1y) > 0:
            aditya_ret_1y = f_1y_rets_available_sorted[aditya_fund_in_cat_1y].values[0]
            local_distance_df_1y.loc[d, 'category Birla scheme return'] = aditya_ret_1y
            
#             behind_birla = f_1y_rets_available_sorted.loc[(aditya_ret_1y > f_1y_rets_available_sorted)].index
            behind_birla = f_1y_rets_available_sorted.loc[(aditya_ret_1y >= f_1y_rets_available_sorted)].index
            local_distance_df_1y.loc[d,'% of AUM in category Birla is outperforming'] = aum2.loc[:d,behind_birla].iloc[-1].sum()/aum2.loc[:d,f_1y_rets_available_sorted.index].iloc[-1].sum()
            local_distance_df_1y.loc[d,'% of funds in category Birla is outperforming'] = len(behind_birla)/len(f_1y_rets_available_sorted)

        local_distance_df_1y.loc[d, 'category top scheme'] = f_1y_rets_available_sorted.index[0]
        local_distance_df_1y.loc[d, 'category bottom scheme'] = f_1y_rets_available_sorted.index[-1]
        if len(f_1y_rets_available_sorted) > 2:
            local_distance_df_1y.loc[d, 'category mean scheme'] = f_1y_rets_available_sorted.index[int(len(f_1y_rets_available_sorted)/2) + (len(f_1y_rets_available_sorted) % 2 > 0)]
        local_distance_df_1y.loc[d, 'category Q1 scheme'] = f_1y_rets_available_sorted[q1_funds_1y].index[-1]
        if len(aditya_fund_in_cat_1y) > 0:
            local_distance_df_1y.loc[d, 'category Birla scheme'] = aditya_fund_in_cat_1y[0]
            ###################### 3 year
        aditya_fund_in_cat_3y = [i for i in f_3y_rets_available_sorted.index if 'Aditya' in i]
        if len(aditya_fund_in_cat_3y) > 1:
            print('More than one ABSL funds in cat ', f)

        local_distance_df_3y.loc[d, 'Category'] = f
        local_distance_df_3y.loc[d, 'category top return'] = f_3y_rets_available_sorted[0]
        local_distance_df_3y.loc[d, 'category bottom return'] = f_3y_rets_available_sorted[-1]
        if len(f_3y_rets_available_sorted) > 2:
            local_distance_df_3y.loc[d, 'category mean return'] = f_3y_rets_available_sorted[int(len(f_3y_rets_available_sorted)/2)]
        if len(f_3y_rets_available_sorted) > 3:
            local_distance_df_3y.loc[d, 'category Q1 return'] = f_3y_rets_available_sorted[q1_funds_3y][-1]
        if len(aditya_fund_in_cat_3y) > 0:
            aditya_ret_3y = f_3y_rets_available_sorted[aditya_fund_in_cat_3y].values[0]
            local_distance_df_3y.loc[d, 'category Birla scheme return'] = aditya_ret_3y
            behind_birla = f_3y_rets_available_sorted.loc[(aditya_ret_3y >= f_3y_rets_available_sorted)].index
            local_distance_df_3y.loc[d,'% of AUM in category Birla is outperforming'] = aum2.loc[:d,behind_birla].iloc[-1].sum()/aum2.loc[:d,f_3y_rets_available_sorted.index].iloc[-1].sum()
            local_distance_df_3y.loc[d,'% of funds in category Birla is outperforming'] = len(behind_birla)/len(f_3y_rets_available_sorted)


        local_distance_df_3y.loc[d, 'category top scheme'] = f_3y_rets_available_sorted.index[0]
        local_distance_df_3y.loc[d, 'category bottom scheme'] = f_3y_rets_available_sorted.index[-1]
        if len(f_3y_rets_available_sorted) > 2:
            local_distance_df_3y.loc[d, 'category mean scheme'] = f_3y_rets_available_sorted.index[int(len(f_3y_rets_available_sorted)/2) + (len(f_3y_rets_available_sorted) % 2 > 0)]
        local_distance_df_3y.loc[d, 'category Q1 scheme'] = f_3y_rets_available_sorted[q1_funds_3y].index[-1]
        if len(aditya_fund_in_cat_3y) > 0:
            local_distance_df_3y.loc[d, 'category Birla scheme'] = aditya_fund_in_cat_3y[0]

    ######################  %tile in returns ########################################
    # 1Y
    f_1y_per = f_1y_rets.rank(axis=1).div(len(f_1y_rets.columns) - pd.isna(f_1y_rets).sum(axis=1), axis=0)
    f_1y_per = f_1y_per.loc[f_1y_rets.index]
    # 3y
    f_3y_per = f_3y_rets.rank(axis=1).div(len(f_3y_rets.columns) - pd.isna(f_3y_rets).sum(axis=1), axis=0)
    f_3y_per = f_3y_per.loc[f_3y_rets.index]

    # Return all the local outputs for this category.
    return {
        'f': f,
        'fs': fs,
        'qy_df_1y': local_qy_df_1y,
        'qy_df_3y': local_qy_df_3y,
        'distance_df_1y': local_distance_df_1y,
        'distance_df_3y': local_distance_df_3y,
        'f_1y_per': f_1y_per,
        'f_3y_per': f_3y_per
        }
####################################################################################################
# Run the processing in parallel over all categories.
# results = Parallel(n_jobs=-1)(delayed(process_category)(f) for f in categories)
results = Parallel(n_jobs=-1)(delayed(process_category)(f) for f in categories_exact)
####################################################################################################
# Merge the results from all categories back into the global DataFrames.
for res in results:
    fs = res['fs']
    # Update the quartile assignments (only for the funds in this category)
    qy_df_1y.loc[:, fs] = res['qy_df_1y']
    qy_df_3y.loc[:, fs] = res['qy_df_3y']
    # Concatenate distance DataFrames.
    distance_df_1y = pd.concat([distance_df_1y, res['distance_df_1y']])
    distance_df_3y = pd.concat([distance_df_3y, res['distance_df_3y']])
    # Merge percentile data.
    percentile_rolling_df_1y = percentile_rolling_df_1y.merge(res['f_1y_per'], left_index=True, right_index=True, how='outer')
    percentile_rolling_df_3y = percentile_rolling_df_3y.merge(res['f_3y_per'], left_index=True, right_index=True, how='outer')
####################################################################################################
# (The remaining code below stays the same.)
percent_in_q1_1y = (qy_df_1y.T == 'q1') * percent_aum_df1.iloc[:, 1:]
percent_in_q2_1y = (qy_df_1y.T == 'q2') * percent_aum_df1.iloc[:, 1:]
percent_in_q3_1y = (qy_df_1y.T == 'q3') * percent_aum_df1.iloc[:, 1:]
percent_in_q4_1y = (qy_df_1y.T == 'q4') * percent_aum_df1.iloc[:, 1:]
percent_in_q1_3y = (qy_df_3y.T == 'q1') * percent_aum_df1.iloc[:, 1:]
percent_in_q2_3y = (qy_df_3y.T == 'q2') * percent_aum_df1.iloc[:, 1:]
percent_in_q3_3y = (qy_df_3y.T == 'q3') * percent_aum_df1.iloc[:, 1:]
percent_in_q4_3y = (qy_df_3y.T == 'q4') * percent_aum_df1.iloc[:, 1:]

percent_in_q1_1y['Quntile'] = '% in Q1'
percent_in_q2_1y['Quntile'] = '% in Q2'
percent_in_q3_1y['Quntile'] = '% in Q3'
percent_in_q4_1y['Quntile'] = '% in Q4'
percent_in_q1_3y['Quntile'] = '% in Q1'
percent_in_q2_3y['Quntile'] = '% in Q2'
percent_in_q3_3y['Quntile'] = '% in Q3'
percent_in_q4_3y['Quntile'] = '% in Q4'

percent_in_q_1y = pd.concat([percent_in_q1_1y, percent_in_q2_1y, percent_in_q3_1y, percent_in_q4_1y])
percent_in_q_3y = pd.concat([percent_in_q1_3y, percent_in_q2_3y, percent_in_q3_3y, percent_in_q4_3y])
percent_in_q_1y = percent_in_q_1y.loc[pd.isna(percent_in_q_1y.iloc[:, :-1]).sum(axis=1) != len(percent_in_q_1y.iloc[:, :-1].columns)]
percent_in_q_3y = percent_in_q_3y.loc[pd.isna(percent_in_q_3y.iloc[:, :-1]).sum(axis=1) != len(percent_in_q_3y.iloc[:, :-1].columns)]
#####################################################################################################
merged_1y = percent_in_q_1y.merge(category_map, left_index=True, right_index=True)
merged_3y = percent_in_q_3y.merge(category_map, left_index=True, right_index=True)
merged_1y['Scheme'] = merged_1y.index
merged_3y['Scheme'] = merged_3y.index
merged_1y['Rolling Window'] = '1 Year'
merged_3y['Rolling Window'] = '3 Year'
merged_1y_grouped = merged_1y.groupby(['Rolling Window','Breakdown', 'Category', 'Scheme Sub Nature', 'fund house', 'Scheme', 'Quntile']).sum()
merged_3y_grouped = merged_3y.groupby(['Rolling Window','Breakdown', 'Category', 'Scheme Sub Nature', 'fund house', 'Scheme', 'Quntile']).sum()
merged_quartile_df = pd.concat([merged_3y_grouped,merged_1y_grouped])
truncate_from = pd.to_datetime('2013-12-31')
merged_quartile_df = merged_quartile_df.loc[:,[i for i in merged_quartile_df.columns if i >= truncate_from]]
####################################################################################################
distance_df_1y['Rolling window'] = '1 Year'
distance_df_3y['Rolling window'] = '3 Year'

distance_df_3y['Birla - category Average'] = distance_df_3y['category Birla scheme return'] - distance_df_3y['category mean return']
distance_df_3y['Birla - Category Top'] = distance_df_3y['category Birla scheme return'] - distance_df_3y['category top return']
distance_df_3y['Birla - Category Bottom'] = distance_df_3y['category Birla scheme return'] - distance_df_3y['category bottom return']
distance_df_3y['Birla - Q1'] = distance_df_3y['category Birla scheme return'] - distance_df_3y['category Q1 return']

distance_df_1y['Birla - category Average'] = distance_df_1y['category Birla scheme return'] - distance_df_1y['category mean return']
distance_df_1y['Birla - Category Top'] = distance_df_1y['category Birla scheme return'] - distance_df_1y['category top return']
distance_df_1y['Birla - Category Bottom'] = distance_df_1y['category Birla scheme return'] - distance_df_1y['category bottom return']
distance_df_1y['Birla - Q1'] = distance_df_1y['category Birla scheme return'] - distance_df_1y['category Q1 return']

# for c in categories : 
#     distance_df_1y.loc[distance_df_1y['Category']==c,'% times Birla with +ve alpha over category average'] = (distance_df_1y.loc[distance_df_1y['Category']==c]['Birla - category Average']>0).rolling(window=250).sum()/250
#     distance_df_3y.loc[distance_df_3y['Category']==c,'% times Birla with +ve alpha over category average'] = (distance_df_3y.loc[distance_df_3y['Category']==c]['Birla - category Average']>0).rolling(window=250).sum()/250 
# merged_distance_df = pd.concat([distance_df_3y,distance_df_1y])
# reorder_cols = ['Rolling window','Category','category top scheme','category bottom scheme', 'category mean scheme',
#                 'category Q1 scheme','category Birla scheme', 'category top return', 'category bottom return','category mean return',
#                 'category Q1 return','category Birla scheme return','Birla - category Average','Birla - Category Top',
#                 'Birla - Category Bottom','Birla - Q1','% times Birla with +ve alpha over category average']
# merged_distance_df = merged_distance_df[reorder_cols]
# merged_distance_df = merged_distance_df.sort_values(by = ['Rolling window','Category'])
####################################################################################################
for c in categories : 
    distance_df_1y.loc[distance_df_1y['Category']==c,'% times Birla with +ve alpha over category average'] = (distance_df_1y.loc[distance_df_1y['Category']==c]['Birla - category Average']>0).rolling(window=250).sum()/250
    distance_df_3y.loc[distance_df_3y['Category']==c,'% times Birla with +ve alpha over category average'] = (distance_df_3y.loc[distance_df_3y['Category']==c]['Birla - category Average']>0).rolling(window=250).sum()/250 
merged_distance_df = pd.concat([distance_df_3y,distance_df_1y])
reorder_cols = columns = ["Rolling window", 
    "Category", 
    "category top return", 
    "category bottom return", 
    "category mean return", 
    "category Q1 return", 
    "category Birla scheme return", 
    "% of AUM in category Birla is outperforming", 
    "% of funds in category Birla is outperforming", 
    "Birla - category Average", 
    "Birla - Category Top", 
    "Birla - Category Bottom", 
    "Birla - Q1", 
    "% times Birla with +ve alpha over category average", 
    "category top scheme", 
    "category bottom scheme", 
    "category mean scheme", 
    "category Q1 scheme", 
    "category Birla scheme"]

merged_distance_df = merged_distance_df[reorder_cols]
merged_distance_df = merged_distance_df.sort_values(by = ['Rolling window','Category'])

In [ ]:
# merged_quartile_df.to_csv('Percent in Q, 3Y, 1Y, All Schemes, All Fund Houses.csv')
# merged_distance_df.to_clipboard()
# merged_quartile_df.to_clipboard()

In [ ]:
################ Aggregation and report generatiom - Sleevewise - used in vantage point etc #############################
df = merged_quartile_df.copy()
df = df.reset_index()

latest_aum_percent = pd.DataFrame(percent_aum_df1.iloc[:,-1]).reset_index()
latest_aum_percent.columns = ['Scheme','% of Total AUM']
df = df.merge(latest_aum_percent,left_on = 'Scheme',right_on = 'Scheme')
df.insert(loc=df.columns.get_loc('Quntile')+1, column='% of Total AUM', value=df.pop('% of Total AUM').values)
# latest_aum_percent_df = df.groupby(['Rolling Window','fund house','Breakdown']).sum()[['% of Total AUM']]
latest_aum_percent_df = df.groupby(['Rolling Window','fund house','Breakdown']).sum(numeric_only=True)[['% of Total AUM']]
##################################################################################

# visual_df = df.groupby(['Rolling Window','fund house','Breakdown','Quntile']).sum().iloc[:,1:]
visual_df = df.groupby(['Rolling Window','fund house','Breakdown','Quntile']).sum(numeric_only=True).iloc[:,1:]
visual_df.index.names = ['Rolling Window', 'fund house', 'Breakdown', 'Quartile']
d_cols = [i for i in visual_df.columns if isinstance(i,pd.Timestamp)]
visual_df = visual_df[d_cols]
# visual_df.columns = pd.to_datetime(visual_df.columns,format = '%d-%m-%Y')
##################################################################################

aum_df_ = pd.DataFrame(percent_aum_df1.iloc[:,1:])
aum_df_.columns = pd.to_datetime(aum_df_.columns)                       
aum_df = category_map.merge(aum_df_[visual_df.columns],left_index=True,right_index=True,how='right')

aum_df = aum_df.loc[aum_df['fund house'].isin(top15)]
aum_df['Quartile'] = '% of AMC AUM'
aum_df1 = aum_df.copy()
aum_df2 = aum_df.copy()
aum_df1['Rolling Window'] = '1 Year'
aum_df2['Rolling Window'] = '3 Year'
aum_df = pd.concat([aum_df1,aum_df2])
# aum_df.columns = pd.to_datetime(aum_df.columns)
##################################################################################
visual_df = visual_df.loc[visual_df.index.get_level_values('fund house').isin(top15),]
# aum_df_total = aum_df.groupby(['Rolling Window','fund house','Breakdown','Quartile']).sum()
aum_df_total = aum_df.groupby(['Rolling Window','fund house','Breakdown','Quartile']).sum(numeric_only=True)

d_cols = [i for i in aum_df_total.columns if isinstance(i,pd.Timestamp)]
aum_df_total = aum_df_total[d_cols]

aum_df_total.columns = pd.to_datetime(aum_df_total.columns)
visual_df.columns = pd.to_datetime(visual_df.columns)

visual_df = pd.concat([visual_df,aum_df_total])
visual_df = visual_df.sort_index(level=['Rolling Window','fund house','Breakdown','Quartile'])
visual_df1 = visual_df.reset_index().groupby(['fund house','Rolling Window','Quartile']).sum(numeric_only = True)

In [ ]:
visual_df1.to_clipboard()

In [ ]:
visual_df1.to_clipboard()

In [ ]:
visual_df.to_clipboard()

In [ ]:
#############################################################################################################################
######################### % in Q1 + Q2 for Birla and whats the rank within brekadown peers ############################################
visual_df_q1q2 = visual_df.reset_index().copy()
visual_df_q1q2 = visual_df_q1q2.loc[visual_df_q1q2['Quartile'].isin(['% in Q1','% in Q2'])]
visual_df_q1q2 = visual_df_q1q2.groupby(['Rolling Window','Breakdown','fund house']).sum()[visual_df.columns]
visual_df_q1q2.columns = pd.to_datetime(visual_df_q1q2.columns)
visual_df_q1q2 = visual_df_q1q2.loc[:,visual_df_q1q2.columns[visual_df_q1q2.columns >= pd.to_datetime('2019-01-01')]]

aum_distri1 = category_map.loc[category_map['fund house'].isin(top15)].merge(percent_aum_df1.iloc[:, 1:], left_index=True, right_index=True, how='left').groupby(['Breakdown','fund house']).sum()
aum_distri1 = aum_distri1[visual_df_q1q2.columns]
visual_df_q1q2_absolute = visual_df_q1q2.copy()
aum_distribution_percent = aum_distri1.copy()
visual_df_q1q2_percent = visual_df_q1q2/aum_distri1

visual_df_q1q2 = visual_df_q1q2.sort_index(level=['Rolling Window','Breakdown','fund house'])
visual_df_q1q2_percent = visual_df_q1q2_percent.sort_index(level=['Rolling Window','Breakdown','fund house'])
visual_df_q1q2_percent_rank = visual_df_q1q2_percent.groupby(['Rolling Window','Breakdown']).rank(ascending=False)

birla_df_q1q2 = visual_df_q1q2.loc[visual_df_q1q2.index.get_level_values('fund house').isin(['Aditya'])]
birla_df_q1q2_percent = visual_df_q1q2_percent.loc[visual_df_q1q2_percent.index.get_level_values('fund house').isin(['Aditya'])]
birla_df_q1q2_percent_rank = visual_df_q1q2_percent_rank.loc[visual_df_q1q2_percent_rank.index.get_level_values('fund house').isin(['Aditya'])]
#############################################################################################################################

######################### % in Q4 for Birla and whats the rank within brekadown peers  ############################################
visual_df_q4 = visual_df.reset_index().copy()
visual_df_q4 = visual_df_q4.loc[visual_df_q4['Quartile'].isin(['% in Q4'])]
visual_df_q4 = visual_df_q4.groupby(['Rolling Window','Breakdown','fund house']).sum()[visual_df.columns]
visual_df_q4.columns = pd.to_datetime(visual_df_q4.columns)
visual_df_q4 = visual_df_q4.loc[:,visual_df_q4.columns[visual_df_q4.columns >= pd.to_datetime('2019-01-01')]]
aum_distri1 = category_map.loc[category_map['fund house'].isin(top15)].merge(percent_aum_df1.iloc[:, 1:], left_index=True, right_index=True, how='left').groupby(['Breakdown','fund house']).sum()
aum_distri1 = aum_distri1[visual_df_q4.columns]
visual_df_q4_absolute = visual_df_q4.copy()
aum_distribution_percent = aum_distri1.copy()
visual_df_q4_percent = visual_df_q4/aum_distri1

visual_df_q4 = visual_df_q4.sort_index(level=['Rolling Window','Breakdown','fund house'])
visual_df_q4_percent = visual_df_q4_percent.sort_index(level=['Rolling Window','Breakdown','fund house'])
visual_df_q4_percent_rank = visual_df_q4_percent.groupby(['Rolling Window','Breakdown']).rank(ascending=False)

birla_df_q4 = visual_df_q4.loc[visual_df_q4.index.get_level_values('fund house').isin(['Aditya'])]
birla_df_q4_percent = visual_df_q4_percent.loc[visual_df_q4_percent.index.get_level_values('fund house').isin(['Aditya'])]
birla_df_q4_percent_rank = visual_df_q4_percent_rank.loc[visual_df_q4_percent_rank.index.get_level_values('fund house').isin(['Aditya'])]

#############################################################################################################################
amc_q1q2_df = visual_df.reset_index()
amc_q1q2_df = amc_q1q2_df.loc[amc_q1q2_df['Quartile'].isin(['% in Q1','% in Q2'])]
amc_q1q2_df = amc_q1q2_df.groupby(['Rolling Window','fund house']).sum()
amc_q1q2_df = amc_q1q2_df.T
#############################################################################################################################
sleeve_aum_distri = aum_distri1.loc[aum_distri1.index.get_level_values('fund house').isin(['Aditya'])].T

In [ ]:
amc_q1q2_df

In [ ]:
######################### CY rets rank #######################
data1 = data.loc[:,category_map.index]

cy_avail = list(set(merged_distance_df.index.year))
cy_avail.sort()
fy_ends = [f'{i}-12-31' for i in cy_avail]

fy_rets = pd.DataFrame()
for i in range(1,len(fy_ends)) : 
	fy = fy_ends[i]
	fy_prev = fy_ends[i-1]
	fy_ret = data1.loc[:fy].iloc[-1]/data1.loc[:fy_prev].iloc[-1]-1
	fy_rets = pd.concat([fy_rets,pd.DataFrame(fy_ret,columns = [fy])],axis=1)
#######################################################################
fy_rank = pd.DataFrame()
fy_percentile = pd.DataFrame()

for c in categories : 
	cfs = list(category_map.loc[category_map['Category'] == c ].index)
	fy_rank_ = fy_rets.loc[cfs].rank(axis=0,ascending=False)
	fy_percentile_ = fy_rets.loc[cfs].rank(axis=0,ascending=True)/fy_rets.loc[cfs].rank(axis=0,ascending=True).max()
	fy_rank = pd.concat([fy_rank,fy_rank_])
	fy_percentile = pd.concat([fy_percentile,fy_percentile_])
fy_rank = category_map.merge(fy_rank,left_index=True,right_index=True)
fy_percentile = category_map.merge(fy_percentile,left_index=True,right_index=True)
fy_rank = fy_rank.sort_values(by = ['Breakdown','Category','fund house'])
fy_percentile = fy_percentile.sort_values(by = ['Breakdown','Category','fund house'])
#######################################################################

In [ ]:
##till 30th June/31 July
fy_ends = [f'{i}-12-31' for i in cy_avail]

fy_ends = fy_ends[:-1] + ['2025-07-31']

fy_rets = pd.DataFrame()
for i in range(1,len(fy_ends)) : 
	fy = fy_ends[i]
	fy_prev = fy_ends[i-1]
	fy_ret = data1.loc[:fy].iloc[-1]/data1.loc[:fy_prev].iloc[-1]-1
	fy_rets = pd.concat([fy_rets,pd.DataFrame(fy_ret,columns = [fy])],axis=1)
#######################################################################
fy_rank = pd.DataFrame()
fy_percentile = pd.DataFrame()

for c in categories : 
	cfs = list(category_map.loc[category_map['Category'] == c ].index)
	fy_rank_ = fy_rets.loc[cfs].rank(axis=0,ascending=False)
	fy_percentile_ = fy_rets.loc[cfs].rank(axis=0,ascending=True)/fy_rets.loc[cfs].rank(axis=0,ascending=True).max()
	fy_rank = pd.concat([fy_rank,fy_rank_])
	fy_percentile = pd.concat([fy_percentile,fy_percentile_])
fy_rank = category_map.merge(fy_rank,left_index=True,right_index=True)
fy_percentile = category_map.merge(fy_percentile,left_index=True,right_index=True)
fy_rank = fy_rank.sort_values(by = ['Breakdown','Category','fund house'])
fy_percentile = fy_percentile.sort_values(by = ['Breakdown','Category','fund house'])
#######################################################################

In [ ]:
##########################################################
############## Export fields ############################
# % of AMC AUM in brerakdown distribution
sleeve_aum_distri.to_clipboard()
# aum_distri1.loc[aum_distri1.index.get_level_values('fund house').isin(['Aditya'])].T.to_clipboard()

### Sleevewise Breakdown --- % of AMC AUM in Q1+Q2
birla_df_q1q2.T.to_clipboard()
birla_df_q1q2_percent.T.to_clipboard()

birla_df_q4.T.to_clipboard()
birla_df_q4_percent.T.to_clipboard()
#############################################################
######### Category overview
merged_distance_df.to_clipboard()

######### Cy percentile and rank #############
fy_percentile.to_clipboard()
fy_rank.to_clipboard()

In [ ]:
df1 = fy_rank.groupby(['fund house','Category']).count().sort_values(by = 'Scheme Sub Nature',ascending=False)
df1.to_clipboard()

In [ ]:
fy_rank.to_clipboard()

In [ ]:
sleeve_aum_distri.to_clipboard()
birla_df_q1q2.T.to_clipboard()

In [ ]:
birla_df_q4.T.to_clipboard()

In [ ]:
birla_df_q1q2_percent.T.to_clipboard()

In [ ]:
birla_df_q4_percent.T.to_clipboard()

In [ ]:
merged_distance_df.to_clipboard()

In [ ]:
category_map.to_clipboard()

In [ ]:
############################################################################################################################################################################
############################################################################################################################################################################

### Scoring 

In [ ]:
################################### Main Script - Exact Peerset From Value Research ############################################################################
###################################  Rolling 250d/750d rets used  ############################################################################
## Final code for exact peer set - Bucket based with Round up

#  Exact peer set
# Avreagre %tile and quantile cutoffs over last 250 days withing category
_1y_rets = data/data.shift(250)-1
_2y_rets = data/data.shift(250*2)-1
_3y_rets = data/data.shift(250*3)-1

_1y_rets_bench = bench_nav/bench_nav.shift(250)-1
_2y_rets_bench = bench_nav/bench_nav.shift(250*2)-1
_3y_rets_bench = bench_nav/bench_nav.shift(250*3)-1

# _1y_alpha = _1y_rets.sub(_1y_rets_bench['NSE500TR Index'],axis=0)
# _2y_alpha = _2y_rets.sub(_2y_rets_bench['NSE500TR Index'],axis=0)
# _3y_alpha = _3y_rets.sub(_3y_rets_bench['NSE500TR Index'],axis=0)

#####
quantile_df = pd.DataFrame()
percentile_rolling_df_1y = pd.DataFrame()
percentile_rolling_df_2y = pd.DataFrame()
percentile_rolling_df_3y = pd.DataFrame()

# qy_df_1y = pd.DataFrame(index = _1y_rets.index,columns = _1y_rets.columns)
# qy_df_3y = pd.DataFrame(index = _3y_rets.index,columns = _3y_rets.columns)

qy_df_1y = pd.DataFrame()
qy_df_3y = pd.DataFrame()

alpha_df_1y = pd.DataFrame()
alpha_df_3y = pd.DataFrame()
  
categories_exact = list(set(category_map_exact['Category']))
avoid_categories = ["Domestic + International",'Multi Index FoF','Other Competitor Thematic/Contra Funds']
categories_exact = [i for i in categories_exact if i not in avoid_categories]
## Sorting so that LargeCap comes after focused category - since both have same peer group
categories_exact.sort()


for f in categories_exact:
    print(f)
    fs = list(category_map_exact.loc[category_map_exact['Category'] ==f].index)

    ### Removing funds fow which NAV was not available
    fs = [i for i in fs if i!= 0]
    ## Removing funds for which NAV was not available - check if multiple are getting out
    fs = [i for i in fs if i in data.columns]

    ####################### Rolling returns #############
#     rets 
    f_1y_rets = _1y_rets[fs]
    f_3y_rets = _3y_rets[fs]
#   Alpha

    f_bench = exact_bench.loc[f,'Benchmark']
    f_alpha_1y = f_1y_rets.sub(_1y_rets_bench[f_bench],axis=0)
    f_alpha_3y = f_3y_rets.sub(_3y_rets_bench[f_bench],axis=0)

    ######################  %tile in returns
#     1Y
    f_1y_per = f_1y_rets.rank(axis=1).div(len(f_1y_rets.columns)-pd.isna(f_1y_rets).sum(axis=1),axis=0)    
    f_1y_per = f_1y_per.loc[f_1y_rets.index]
    f_1y_per.columns = pd.MultiIndex.from_tuples(list(zip([f]*len(f_1y_per.columns),f_1y_per.columns)))
    # percentile_rolling_df_1y = percentile_rolling_df_1y.merge(f_1y_per,left_index=True,right_index=True,how='outer')
    percentile_rolling_df_1y = pd.concat([percentile_rolling_df_1y,f_1y_per],axis=1)
#     3y
    f_3y_per = f_3y_rets.rank(axis=1).div(len(f_3y_rets.columns)-pd.isna(f_3y_rets).sum(axis=1),axis=0)    
    f_3y_per = f_3y_per.loc[f_3y_rets.index]
    f_3y_per.columns = pd.MultiIndex.from_tuples(list(zip([f]*len(f_3y_per.columns),f_3y_per.columns)))
    # percentile_rolling_df_3y = percentile_rolling_df_3y.merge(f_3y_per,left_index=True,right_index=True,how='outer')
    percentile_rolling_df_3y = pd.concat([percentile_rolling_df_3y,f_3y_per],axis=1)

    ##########################################################################
    ############################################################################################################################

    date_list = f_3y_rets.loc[pd.isna(f_3y_rets).sum(axis=1) != len(f_3y_rets.columns)].index
    qy_df_1y_ = pd.DataFrame(index = date_list,columns = fs)
    qy_df_3y_ = pd.DataFrame(index = date_list,columns = fs)

    for d in date_list : 
        ######### 1 year rets quartile 
        f_1y_rets_available = f_1y_rets.loc[d][pd.isna(f_1y_rets.loc[d])==False]
        f_1y_rets_available_sorted = f_1y_rets_available.sort_values(ascending=False)

        ## Rounding up when number of funds is not divisible for 4
        items_each_bucket = len(f_1y_rets_available)//4
        extra_items = len(f_1y_rets_available)%4
        bucket_items = [items_each_bucket]*4
        for i in range(1,extra_items+1) : 
            bucket_items[i-1] = bucket_items[i]+1

        q1_funds_1y = list(f_1y_rets_available_sorted[:bucket_items[0]].index)
        q2_funds_1y = list(f_1y_rets_available_sorted[bucket_items[0]:][:bucket_items[1]].index)
        q3_funds_1y = list(f_1y_rets_available_sorted[bucket_items[0]+bucket_items[1]:][:bucket_items[2]].index)
        q4_funds_1y = list(f_1y_rets_available_sorted[bucket_items[0]+bucket_items[1]+bucket_items[2]:][:bucket_items[3]].index)

        ######### 3 year rets quartile ###########
        f_3y_rets_available = f_3y_rets.loc[d][pd.isna(f_3y_rets.loc[d])==False]
        f_3y_rets_available_sorted = f_3y_rets_available.sort_values(ascending=False)

        ## Rounding up when number of funds is not divisible for 4
        items_each_bucket = len(f_3y_rets_available)//4
        extra_items = len(f_3y_rets_available)%4
        bucket_items = [items_each_bucket]*4
        for i in range(1,extra_items+1) : 
            bucket_items[i-1] = bucket_items[i]+1

        q1_funds_3y = list(f_3y_rets_available_sorted[:bucket_items[0]].index)
        q2_funds_3y = list(f_3y_rets_available_sorted[bucket_items[0]:][:bucket_items[1]].index)
        q3_funds_3y = list(f_3y_rets_available_sorted[bucket_items[0]+bucket_items[1]:][:bucket_items[2]].index)
        q4_funds_3y = list(f_3y_rets_available_sorted[bucket_items[0]+bucket_items[1]+bucket_items[2]:][:bucket_items[3]].index)

        ########################## Assigning quartiles ###############

        qy_df_1y_.loc[d,q1_funds_1y] = 'q1'
        qy_df_1y_.loc[d,q2_funds_1y] = 'q2'
        qy_df_1y_.loc[d,q3_funds_1y] = 'q3'
        qy_df_1y_.loc[d,q4_funds_1y] = 'q4'

        qy_df_3y_.loc[d,q1_funds_3y] = 'q1'
        qy_df_3y_.loc[d,q2_funds_3y] = 'q2'
        qy_df_3y_.loc[d,q3_funds_3y] = 'q3'
        qy_df_3y_.loc[d,q4_funds_3y] = 'q4'

    qy_df_1y_.columns = pd.MultiIndex.from_tuples(list(zip([f]*len(qy_df_1y_.columns),qy_df_1y_.columns)))
    qy_df_3y_.columns = pd.MultiIndex.from_tuples(list(zip([f]*len(qy_df_3y_.columns),qy_df_3y_.columns)))

    alpha_df_1y_ = f_alpha_1y.loc[date_list]
    alpha_df_3y_ = f_alpha_3y.loc[date_list]
    alpha_df_1y_.columns = pd.MultiIndex.from_tuples(list(zip([f]*len(alpha_df_1y_.columns),alpha_df_1y_.columns)))
    alpha_df_3y_.columns = pd.MultiIndex.from_tuples(list(zip([f]*len(alpha_df_3y_.columns),alpha_df_3y_.columns)))

    # qy_df_1y = qy_df_1y.merge(qy_df_1y_,left_index=True,right_index=True,how='outer')
    # qy_df_3y = qy_df_3y.merge(qy_df_3y_,left_index=True,right_index=True,how='outer')

    qy_df_1y = pd.concat([qy_df_1y,qy_df_1y_],axis=1)
    qy_df_3y = pd.concat([qy_df_3y,qy_df_3y_],axis=1)
    
    alpha_df_1y = pd.concat([alpha_df_1y,alpha_df_1y_],axis=1)
    alpha_df_3y = pd.concat([alpha_df_3y,alpha_df_3y_],axis=1)
    

# percentile_rolling_df_1y = percentile_rolling_df_1y.ffill()
# percentile_rolling_df_2y = percentile_rolling_df_2y.ffill()
# percentile_rolling_df_3y = percentile_rolling_df_3y.ffill()

In [ ]:
# t = data.index[-1]
# t_1y = data.loc[:f'{t.year-1}-{t.month}-{t.day}'].index[-1]
# t_3y = data.loc[:f'{t.year-3}-{t.month}-{t.day}'].index[-1]

# rets_exact_1y = data.loc[t]/data.loc[t_1y]-1
# rets_exact_3y = (data.loc[t]/data.loc[t_3y])**(1/3)-1
# rets_exact_bench_1y = bench_nav.loc[t]/bench_nav.loc[t_1y]-1
# rets_exact_bench_3y = (bench_nav.loc[t]/bench_nav.loc[t_3y])**(1/3)-1

In [ ]:
################################### Main Script - Exact Peerset From Value Research ############################################################################
###################################  Point to Point Calendar Year 1Y/3Y rets used  ############################################################################
## Final code for exact peer set - Bucket based with Round up

###### Exact Calendar Date wise 1Y/3Y rolling returns for Schemes and Benchmarks for Scoring Calculations ##################
_1y_rets = pd.DataFrame()
_3y_rets = pd.DataFrame()

_1y_rets_bench = pd.DataFrame()
_3y_rets_bench = pd.DataFrame()

#########################################################################################################
###### Calendar Date wise 1Y and 3Y rolling returns ############
### Starting 3 years CYwise ahead from first data point #############
start_date = data.loc[f'{data.index[0].year+3}-{data.index[0].month}-{data.index[0].day}':].index[0]
for t in data.loc[start_date:].index : 

    if (t.month == 2) & (t.day == 29) : 
        t_1y_ = pd.to_datetime(f'{t.year-1}-{t.month}-{t.day-1}')
        t_3y_ = pd.to_datetime(f'{t.year-3}-{t.month}-{t.day-1}')       
    else : 
        t_1y_ = pd.to_datetime(f'{t.year-1}-{t.month}-{t.day}')
        t_3y_ = pd.to_datetime(f'{t.year-3}-{t.month}-{t.day}')


    t_1y = data.loc[:t_1y_].index[-1]
    t_3y = data.loc[:t_3y_].index[-1]
    
    rets_exact_1y = data.loc[t]/data.loc[t_1y]-1
    rets_exact_3y = (data.loc[t]/data.loc[t_3y])**(1/3)-1
    rets_exact_bench_1y = bench_nav.loc[t]/bench_nav.loc[t_1y]-1
    rets_exact_bench_3y = (bench_nav.loc[t]/bench_nav.loc[t_3y])**(1/3)-1

    _1y_rets = pd.concat([_1y_rets,pd.DataFrame(rets_exact_1y,columns = [t]).T])
    _3y_rets = pd.concat([_3y_rets,pd.DataFrame(rets_exact_3y,columns = [t]).T])

    _1y_rets_bench = pd.concat([_1y_rets_bench,pd.DataFrame(rets_exact_bench_1y,columns = [t]).T])
    _3y_rets_bench = pd.concat([_3y_rets_bench,pd.DataFrame(rets_exact_bench_3y,columns = [t]).T])
    
#########################################################################################################
quantile_df = pd.DataFrame()
percentile_rolling_df_1y = pd.DataFrame()
percentile_rolling_df_2y = pd.DataFrame()
percentile_rolling_df_3y = pd.DataFrame()

# qy_df_1y = pd.DataFrame(index = _1y_rets.index,columns = _1y_rets.columns)
# qy_df_3y = pd.DataFrame(index = _3y_rets.index,columns = _3y_rets.columns)

qy_df_1y = pd.DataFrame()
qy_df_3y = pd.DataFrame()

alpha_df_1y = pd.DataFrame()
alpha_df_3y = pd.DataFrame()
  
categories_exact = list(set(category_map_exact['Category']))
avoid_categories = ["Domestic + International",'Multi Index FoF','Other Competitor Thematic/Contra Funds']
categories_exact = [i for i in categories_exact if i not in avoid_categories]
## Sorting so that LargeCap comes after focused category - since both have same peer group
categories_exact.sort()


for f in categories_exact:
    print(f)
    fs = list(category_map_exact.loc[category_map_exact['Category'] ==f].index)

    ### Removing funds fow which NAV was not available
    fs = [i for i in fs if i!= 0]
    ## Removing funds for which NAV was not available - check if multiple are getting out
    fs = [i for i in fs if i in data.columns]

    ####################### Rolling returns #############
#     rets 
    f_1y_rets = _1y_rets[fs]
    f_3y_rets = _3y_rets[fs]
#   Alpha

    f_bench = exact_bench.loc[f,'Benchmark']
    f_alpha_1y = f_1y_rets.sub(_1y_rets_bench[f_bench],axis=0)
    f_alpha_3y = f_3y_rets.sub(_3y_rets_bench[f_bench],axis=0)

    ######################  %tile in returns
#     1Y
    f_1y_per = f_1y_rets.rank(axis=1).div(len(f_1y_rets.columns)-pd.isna(f_1y_rets).sum(axis=1),axis=0)    
    f_1y_per = f_1y_per.loc[f_1y_rets.index]
    f_1y_per.columns = pd.MultiIndex.from_tuples(list(zip([f]*len(f_1y_per.columns),f_1y_per.columns)))
    # percentile_rolling_df_1y = percentile_rolling_df_1y.merge(f_1y_per,left_index=True,right_index=True,how='outer')
    percentile_rolling_df_1y = pd.concat([percentile_rolling_df_1y,f_1y_per],axis=1)
#     3y
    f_3y_per = f_3y_rets.rank(axis=1).div(len(f_3y_rets.columns)-pd.isna(f_3y_rets).sum(axis=1),axis=0)    
    f_3y_per = f_3y_per.loc[f_3y_rets.index]
    f_3y_per.columns = pd.MultiIndex.from_tuples(list(zip([f]*len(f_3y_per.columns),f_3y_per.columns)))
    # percentile_rolling_df_3y = percentile_rolling_df_3y.merge(f_3y_per,left_index=True,right_index=True,how='outer')
    percentile_rolling_df_3y = pd.concat([percentile_rolling_df_3y,f_3y_per],axis=1)

    ##########################################################################
    ############################################################################################################################

    date_list = f_3y_rets.loc[pd.isna(f_3y_rets).sum(axis=1) != len(f_3y_rets.columns)].index
    qy_df_1y_ = pd.DataFrame(index = date_list,columns = fs)
    qy_df_3y_ = pd.DataFrame(index = date_list,columns = fs)

    for d in date_list : 
        ######### 1 year rets quartile 
        f_1y_rets_available = f_1y_rets.loc[d][pd.isna(f_1y_rets.loc[d])==False]
        f_1y_rets_available_sorted = f_1y_rets_available.sort_values(ascending=False)

        ## Rounding up when number of funds is not divisible for 4
        items_each_bucket = len(f_1y_rets_available)//4
        extra_items = len(f_1y_rets_available)%4
        bucket_items = [items_each_bucket]*4
        for i in range(1,extra_items+1) : 
            bucket_items[i-1] = bucket_items[i]+1

        q1_funds_1y = list(f_1y_rets_available_sorted[:bucket_items[0]].index)
        q2_funds_1y = list(f_1y_rets_available_sorted[bucket_items[0]:][:bucket_items[1]].index)
        q3_funds_1y = list(f_1y_rets_available_sorted[bucket_items[0]+bucket_items[1]:][:bucket_items[2]].index)
        q4_funds_1y = list(f_1y_rets_available_sorted[bucket_items[0]+bucket_items[1]+bucket_items[2]:][:bucket_items[3]].index)

        ######### 3 year rets quartile ###########
        f_3y_rets_available = f_3y_rets.loc[d][pd.isna(f_3y_rets.loc[d])==False]
        f_3y_rets_available_sorted = f_3y_rets_available.sort_values(ascending=False)

        ## Rounding up when number of funds is not divisible for 4
        items_each_bucket = len(f_3y_rets_available)//4
        extra_items = len(f_3y_rets_available)%4
        bucket_items = [items_each_bucket]*4
        for i in range(1,extra_items+1) : 
            bucket_items[i-1] = bucket_items[i]+1

        q1_funds_3y = list(f_3y_rets_available_sorted[:bucket_items[0]].index)
        q2_funds_3y = list(f_3y_rets_available_sorted[bucket_items[0]:][:bucket_items[1]].index)
        q3_funds_3y = list(f_3y_rets_available_sorted[bucket_items[0]+bucket_items[1]:][:bucket_items[2]].index)
        q4_funds_3y = list(f_3y_rets_available_sorted[bucket_items[0]+bucket_items[1]+bucket_items[2]:][:bucket_items[3]].index)

        ########################## Assigning quartiles ###############

        qy_df_1y_.loc[d,q1_funds_1y] = 'q1'
        qy_df_1y_.loc[d,q2_funds_1y] = 'q2'
        qy_df_1y_.loc[d,q3_funds_1y] = 'q3'
        qy_df_1y_.loc[d,q4_funds_1y] = 'q4'

        qy_df_3y_.loc[d,q1_funds_3y] = 'q1'
        qy_df_3y_.loc[d,q2_funds_3y] = 'q2'
        qy_df_3y_.loc[d,q3_funds_3y] = 'q3'
        qy_df_3y_.loc[d,q4_funds_3y] = 'q4'

    qy_df_1y_.columns = pd.MultiIndex.from_tuples(list(zip([f]*len(qy_df_1y_.columns),qy_df_1y_.columns)))
    qy_df_3y_.columns = pd.MultiIndex.from_tuples(list(zip([f]*len(qy_df_3y_.columns),qy_df_3y_.columns)))

    alpha_df_1y_ = f_alpha_1y.loc[date_list]
    alpha_df_3y_ = f_alpha_3y.loc[date_list]
    alpha_df_1y_.columns = pd.MultiIndex.from_tuples(list(zip([f]*len(alpha_df_1y_.columns),alpha_df_1y_.columns)))
    alpha_df_3y_.columns = pd.MultiIndex.from_tuples(list(zip([f]*len(alpha_df_3y_.columns),alpha_df_3y_.columns)))

    # qy_df_1y = qy_df_1y.merge(qy_df_1y_,left_index=True,right_index=True,how='outer')
    # qy_df_3y = qy_df_3y.merge(qy_df_3y_,left_index=True,right_index=True,how='outer')

    qy_df_1y = pd.concat([qy_df_1y,qy_df_1y_],axis=1)
    qy_df_3y = pd.concat([qy_df_3y,qy_df_3y_],axis=1)
    
    alpha_df_1y = pd.concat([alpha_df_1y,alpha_df_1y_],axis=1)
    alpha_df_3y = pd.concat([alpha_df_3y,alpha_df_3y_],axis=1)
    

# percentile_rolling_df_1y = percentile_rolling_df_1y.ffill()
# percentile_rolling_df_2y = percentile_rolling_df_2y.ffill()
# percentile_rolling_df_3y = percentile_rolling_df_3y.ffill()

In [ ]:
### Score for Fund scheme
df_1y = qy_df_1y.replace('q1',5).replace('q2',4).replace('q3',3).replace('q4',2).replace("",np.nan).ffill()
## 3Y have lower score, but get +1 score if beat the benchmark
df_3y = qy_df_3y.replace('q1',4).replace('q2',3).replace('q3',2).replace('q4',1).replace("",np.nan).ffill()
df_3y_raw = df_3y.copy()

# ### Rewarding the score by 1 if fund has outperformed compared to benchmark in 2Y, 3Y 

# #### For all Schemes
# df_3y = df_3y + (1)*(_3y_alpha[df_3y.columns] > 0).astype(int)

#### For exact value research peers
df_3y = df_3y + (1)*(alpha_df_3y[df_3y.columns]>0).astype(int)

# ### Weighted score 
score_df = 0.8*df_1y + 0.2*df_3y 
# # score_df = 1*df_1y
# # score_df = 1*df_3y

### if a category is <3 years old then no funds will have any score - dropping that category, eg. conglomerate
if score_df[f].sum().sum()==0 : 
    score_df = score_df.drop(f,axis=1)

In [ ]:
###### Exports on March 10, 2026
_1y_rets.loc[date_list,df_1y.columns.get_level_values(1)].to_clipboard()
_3y_rets.loc[date_list,df_3y.columns.get_level_values(1)].to_clipboard()

df_1y.to_clipboard()
df_3y.to_clipboard()
df_3y_raw .to_clipboard()

unique_peers = list(category_map_exact.loc[category_map_exact['Category'].isin(categories_exact)].index)
# _1y_rets[unique_peers].to_clipboard()

pd.DataFrame(_3y_rets_bench.columns).to_clipboard()
alpha_df_3y.to_clipboard()

score_df.to_clipboard()

In [ ]:
_1y_rets.loc[date_list,df_1y.columns.get_level_values(1)].iloc[-1918:].to_clipboard()

In [ ]:
_3y_rets.loc[date_list,df_3y.columns.get_level_values(1)].iloc[-1918:].to_clipboard()

In [ ]:
_3y_rets_bench.loc[date_list].iloc[-1918:].to_clipboard()

In [ ]:
alpha_df_3y.loc[date_list].iloc[-1918:].to_clipboard()

In [ ]:
df_1y.loc[date_list].iloc[-1918:].to_clipboard()

In [ ]:
df_3y_raw.loc[date_list].iloc[-1918:].to_clipboard()

In [ ]:
df_3y.loc[date_list].iloc[-1918:].to_clipboard()

In [ ]:
score_df.loc[date_list].iloc[-1918:].to_clipboard()

In [ ]:
################################################################################################################################################################################

In [ ]:
percentile_rolling_df_3y.loc[date_list].to_clipboard()

In [ ]:
score_df.to_clipboard()

In [ ]:
qy_df_score.to_clipboard()

In [ ]:
qy_df_1y.to_clipboard()

In [ ]:
# ###### Exports 
# _1y_rets.loc[date_list,df_1y.columns.get_level_values(1)].to_clipboard()
# _3y_rets.loc[date_list,df_3y.columns.get_level_values(1)].to_clipboard()

# df_1y.to_clipboard()
# df_3y.to_clipboard()
# df_3y_raw .to_clipboard()

# unique_peers = list(category_map_exact.loc[category_map_exact['Category'].isin(categories_exact)].index)
# # _1y_rets[unique_peers].to_clipboard()
# # _1y_rets.loc[score_df.index,score_df.columns.get_level_values(1)].to_clipboard()
# # _3y_rets.loc[score_df.index,score_df.columns.get_level_values(1)].to_clipboard()

# pd.DataFrame(_3y_rets_bench.columns).to_clipboard()
# alpha_df_3y.to_clipboard()


In [ ]:
########### Using composite score to get Q1, Q2, Q3, Q4 using bucketing - with roundup #################
qy_df_score = pd.DataFrame()
score_per  = pd.DataFrame()

date_list = score_df.loc[pd.isna(score_df).sum(axis=1) != len(score_df.columns)].index

# for f in categories_exact:
for f in list(set(score_df.columns.get_level_values(0))):
    print(f)
    fs = list(category_map_exact.loc[category_map_exact['Category'] ==f].index)
    
    ### Removing funds fow which NAV was not available
    fs = [i for i in fs if i!= 0]
    ## Removing funds for which NAV was not available - check if multiple are getting out
    fs = [i for i in fs if i in data.columns]

    ##########################################################################
    qy_df_score_ = pd.DataFrame()
    score_per_  = pd.DataFrame()
    for d in date_list : 
        ######### 1 year rets quartile 
        score_available = score_df.loc[d,(f,fs)][pd.isna(score_df.loc[d,(f,fs)])==False]
        if len(score_available) >0 : 
            score_available = score_available[f]
            score_available_sorted = score_available.sort_values(ascending=False)

            ## Rounding up when number of funds is not divisible for 4
            items_each_bucket = len(score_available)//4
            extra_items = len(score_available)%4
            bucket_items = [items_each_bucket]*4
            for i in range(1,extra_items+1) : 
                bucket_items[i-1] = bucket_items[i]+1

            q1_funds_score = list(score_available_sorted[:bucket_items[0]].index)
            q2_funds_score = list(score_available_sorted[bucket_items[0]:][:bucket_items[1]].index)
            q3_funds_score = list(score_available_sorted[bucket_items[0]+bucket_items[1]:][:bucket_items[2]].index)
            q4_funds_score = list(score_available_sorted[bucket_items[0]+bucket_items[1]+bucket_items[2]:][:bucket_items[3]].index)

            ########################## Assigning quartiles  ##############

            qy_df_score_.loc[d,q1_funds_score] = 'q1'
            qy_df_score_.loc[d,q2_funds_score] = 'q2'
            qy_df_score_.loc[d,q3_funds_score] = 'q3'
            qy_df_score_.loc[d,q4_funds_score] = 'q4'
            score_per__ = pd.DataFrame(score_available_sorted.rank().div(len(score_available_sorted)-pd.isna(score_available_sorted).sum())).T
            score_per_ = pd.concat([score_per_,score_per__])

    if len(qy_df_score_) >0 :    
        qy_df_score_.columns = pd.MultiIndex.from_tuples(list(zip([f]*len(qy_df_score_.columns),qy_df_score_.columns)))
        qy_df_score = pd.concat([qy_df_score,qy_df_score_],axis=1)

        score_per_.columns = pd.MultiIndex.from_tuples(list(zip([f]*len(score_per_.columns),score_per_.columns)))
        score_per = pd.concat([score_per,score_per_],axis=1)


In [ ]:
############# % of schemes and % of AUM in Q1 + Q2 for each AMC #############################################################
################################################## using bucketing in ranking again ############################

In [ ]:
############################### What % of AUM in in Q1 to Q4 -Composite Score Based - Dropping Bla Bhavishya & retirement 40 due to repeated peersets ###################################
############################### for exact Value Research peer based ###################################
###############################
qy_df_score_ = qy_df_score.loc[:,qy_df_score.columns.get_level_values(0).isin(reperated_peer_category)==False]

percent_aum_df_ = percent_aum_df.loc[qy_df_score_.index,qy_df_score_.columns.get_level_values(1)]
percent_aum_df_.columns = qy_df_score_.columns
###############################
q1_ = ((qy_df_score_ == 'q1')*percent_aum_df_)[qy_df_score_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')
q2_ = ((qy_df_score_ == 'q2')*percent_aum_df_)[qy_df_score_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')
q3_ = ((qy_df_score_ == 'q3')*percent_aum_df_)[qy_df_score_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')
q4_ = ((qy_df_score_ == 'q4')*percent_aum_df_)[qy_df_score_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')

q1_aum_per = category_map[['fund house']].merge(q1_,left_index=True,right_index=True,how='right')
q2_aum_per = category_map[['fund house']].merge(q2_,left_index=True,right_index=True,how='right')
q3_aum_per = category_map[['fund house']].merge(q3_,left_index=True,right_index=True,how='right')
q4_aum_per = category_map[['fund house']].merge(q4_,left_index=True,right_index=True,how='right')
###############################################
percent_aum_in_q1 = q1_aum_per.groupby('fund house').sum()
percent_aum_in_q2 = q2_aum_per.groupby('fund house').sum()
percent_aum_in_q3 = q3_aum_per.groupby('fund house').sum()
percent_aum_in_q4 = q4_aum_per.groupby('fund house').sum()
###############################################
no_of_schemes = category_map[['fund house']].merge((pd.isna(qy_df_score_) == False).T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
exposure_aum = category_map[['fund house']].merge(((pd.isna(qy_df_score_) == False)*percent_aum_df_).T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()

aum_weighted_percentile = category_map[['fund house']].merge((score_per[qy_df_score_.columns]*percent_aum_df_).T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum().T
###############################################

schemes_in_q1 = category_map[['fund house']].merge((qy_df_score_ == 'q1')[qy_df_score_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
schemes_in_q2 = category_map[['fund house']].merge((qy_df_score_ == 'q2')[qy_df_score_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
schemes_in_q3 = category_map[['fund house']].merge((qy_df_score_ == 'q3')[qy_df_score_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
schemes_in_q4 = category_map[['fund house']].merge((qy_df_score_ == 'q4')[qy_df_score_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()

percent_schemes_in_q1 = schemes_in_q1/no_of_schemes
percent_schemes_in_q2 = schemes_in_q2/no_of_schemes
percent_schemes_in_q3 = schemes_in_q3/no_of_schemes
percent_schemes_in_q4 = schemes_in_q4/no_of_schemes
###############################################


In [ ]:
############################### What % of AUM in in Q1 to Q4 - 1Y score - Dropping Bla Bhavishya & retirement 40 due to repeated peersets ###################################
############################### for exact Value Research peer based ###################################
###############################
qy_df_1y_ = qy_df_1y.loc[:,qy_df_1y.columns.get_level_values(0).isin(reperated_peer_category)==False]

percent_aum_df_ = percent_aum_df.loc[qy_df_1y_.index,qy_df_1y_.columns.get_level_values(1)]
percent_aum_df_.columns = qy_df_1y_.columns
###############################
q1_ = ((qy_df_1y_ == 'q1')*percent_aum_df_)[qy_df_1y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')
q2_ = ((qy_df_1y_ == 'q2')*percent_aum_df_)[qy_df_1y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')
q3_ = ((qy_df_1y_ == 'q3')*percent_aum_df_)[qy_df_1y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')
q4_ = ((qy_df_1y_ == 'q4')*percent_aum_df_)[qy_df_1y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')

q1_aum_per = category_map[['fund house']].merge(q1_,left_index=True,right_index=True,how='right')
q2_aum_per = category_map[['fund house']].merge(q2_,left_index=True,right_index=True,how='right')
q3_aum_per = category_map[['fund house']].merge(q3_,left_index=True,right_index=True,how='right')
q4_aum_per = category_map[['fund house']].merge(q4_,left_index=True,right_index=True,how='right')
###############################################
percent_aum_in_q1 = q1_aum_per.groupby('fund house').sum()
percent_aum_in_q2 = q2_aum_per.groupby('fund house').sum()
percent_aum_in_q3 = q3_aum_per.groupby('fund house').sum()
percent_aum_in_q4 = q4_aum_per.groupby('fund house').sum()
###############################################
no_of_schemes = category_map[['fund house']].merge((pd.isna(qy_df_1y_) == False).T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
exposure_aum = category_map[['fund house']].merge(((pd.isna(qy_df_1y_) == False)*percent_aum_df_).T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()

aum_weighted_percentile = category_map[['fund house']].merge((percentile_rolling_df_1y[qy_df_1y_.columns]*percent_aum_df_).T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum().T
###############################################

schemes_in_q1 = category_map[['fund house']].merge((qy_df_1y_ == 'q1')[qy_df_1y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
schemes_in_q2 = category_map[['fund house']].merge((qy_df_1y_ == 'q2')[qy_df_1y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
schemes_in_q3 = category_map[['fund house']].merge((qy_df_1y_ == 'q3')[qy_df_1y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
schemes_in_q4 = category_map[['fund house']].merge((qy_df_1y_ == 'q4')[qy_df_1y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()

percent_schemes_in_q1 = schemes_in_q1/no_of_schemes
percent_schemes_in_q2 = schemes_in_q2/no_of_schemes
percent_schemes_in_q3 = schemes_in_q3/no_of_schemes
percent_schemes_in_q4 = schemes_in_q4/no_of_schemes
###############################################

In [ ]:
############################### What % of AUM in in Q1 to Q4 - 3Y score - Dropping Bla Bhavishya & retirement 40 due to repeated peersets ###################################
############################### for exact Value Research peer based ###################################
###############################
qy_df_3y_ = qy_df_3y.loc[:,qy_df_3y.columns.get_level_values(0).isin(reperated_peer_category)==False]

percent_aum_df_ = percent_aum_df.loc[qy_df_3y_.index,qy_df_3y_.columns.get_level_values(1)]
percent_aum_df_.columns = qy_df_3y_.columns
###############################
q1_ = ((qy_df_3y_ == 'q1')*percent_aum_df_)[qy_df_3y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')
q2_ = ((qy_df_3y_ == 'q2')*percent_aum_df_)[qy_df_3y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')
q3_ = ((qy_df_3y_ == 'q3')*percent_aum_df_)[qy_df_3y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')
q4_ = ((qy_df_3y_ == 'q4')*percent_aum_df_)[qy_df_3y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')

q1_aum_per = category_map[['fund house']].merge(q1_,left_index=True,right_index=True,how='right')
q2_aum_per = category_map[['fund house']].merge(q2_,left_index=True,right_index=True,how='right')
q3_aum_per = category_map[['fund house']].merge(q3_,left_index=True,right_index=True,how='right')
q4_aum_per = category_map[['fund house']].merge(q4_,left_index=True,right_index=True,how='right')
###############################################
percent_aum_in_q1 = q1_aum_per.groupby('fund house').sum()
percent_aum_in_q2 = q2_aum_per.groupby('fund house').sum()
percent_aum_in_q3 = q3_aum_per.groupby('fund house').sum()
percent_aum_in_q4 = q4_aum_per.groupby('fund house').sum()
###############################################
no_of_schemes = category_map[['fund house']].merge((pd.isna(qy_df_3y_) == False).T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
exposure_aum = category_map[['fund house']].merge(((pd.isna(qy_df_3y_) == False)*percent_aum_df_).T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()

aum_weighted_percentile = category_map[['fund house']].merge((percentile_rolling_df_3y[qy_df_3y_.columns]*percent_aum_df_).T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum().T
###############################################

schemes_in_q1 = category_map[['fund house']].merge((qy_df_3y_ == 'q1')[qy_df_3y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
schemes_in_q2 = category_map[['fund house']].merge((qy_df_3y_ == 'q2')[qy_df_3y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
schemes_in_q3 = category_map[['fund house']].merge((qy_df_3y_ == 'q3')[qy_df_3y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
schemes_in_q4 = category_map[['fund house']].merge((qy_df_3y_ == 'q4')[qy_df_3y_.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()

percent_schemes_in_q1 = schemes_in_q1/no_of_schemes
percent_schemes_in_q2 = schemes_in_q2/no_of_schemes
percent_schemes_in_q3 = schemes_in_q3/no_of_schemes
percent_schemes_in_q4 = schemes_in_q4/no_of_schemes
###############################################


In [ ]:
aum_weighted_percentile.to_clipboard()

In [ ]:
percent_schemes_in_q4.T.to_clipboard()

In [ ]:
exposure_aum.T.to_clipboard()

In [ ]:
aum_weighted_percentile.to_clipboard()

In [ ]:
# ############################### What % of AUM in in Q1 to Q4 - will be >100% for major peers eg ICICI since repeated peers in Bal Bhavishya & retirement40 ###############################
# ############################### for exact Value Research peer based ###################################
# ###############################
# percent_aum_df_ = percent_aum_df.loc[qy_df_score.index,qy_df_score.columns.get_level_values(1)]
# percent_aum_df_.columns = qy_df_score.columns
# ###############################
# q1_ = ((qy_df_score == 'q1')*percent_aum_df_)[qy_df_score.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')
# q2_ = ((qy_df_score == 'q2')*percent_aum_df_)[qy_df_score.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')
# q3_ = ((qy_df_score == 'q3')*percent_aum_df_)[qy_df_score.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')
# q4_ = ((qy_df_score == 'q4')*percent_aum_df_)[qy_df_score.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1')

# q1_aum_per = category_map[['fund house']].merge(q1_,left_index=True,right_index=True,how='right')
# q2_aum_per = category_map[['fund house']].merge(q2_,left_index=True,right_index=True,how='right')
# q3_aum_per = category_map[['fund house']].merge(q3_,left_index=True,right_index=True,how='right')
# q4_aum_per = category_map[['fund house']].merge(q4_,left_index=True,right_index=True,how='right')
# ###############################################
# percent_aum_in_q1 = q1_aum_per.groupby('fund house').sum()
# percent_aum_in_q2 = q2_aum_per.groupby('fund house').sum()
# percent_aum_in_q3 = q3_aum_per.groupby('fund house').sum()
# percent_aum_in_q4 = q4_aum_per.groupby('fund house').sum()
# ###############################################
# no_of_schemes = category_map[['fund house']].merge((pd.isna(qy_df_score) == False).T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
# exposure_aum = category_map[['fund house']].merge(((pd.isna(qy_df_score) == False)*percent_aum_df_).T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()


# aum_weighted_percentile = category_map[['fund house']].merge((score_per*percent_aum_df_).T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum().T
# ###############################################

# schemes_in_q1 = category_map[['fund house']].merge((qy_df_score == 'q1')[qy_df_score.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
# schemes_in_q2 = category_map[['fund house']].merge((qy_df_score == 'q2')[qy_df_score.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
# schemes_in_q3 = category_map[['fund house']].merge((qy_df_score == 'q3')[qy_df_score.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()
# schemes_in_q4 = category_map[['fund house']].merge((qy_df_score == 'q4')[qy_df_score.columns].T.reset_index().drop('level_0',axis=1).set_index('level_1'),left_index=True,right_index=True,how='right').groupby('fund house').sum()

# percent_schemes_in_q1 = schemes_in_q1/no_of_schemes
# percent_schemes_in_q2 = schemes_in_q2/no_of_schemes
# percent_schemes_in_q3 = schemes_in_q3/no_of_schemes
# percent_schemes_in_q4 = schemes_in_q4/no_of_schemes
# ###############################################

In [ ]:
exposure_aum.T.to_clipboard()

In [ ]:
aum_weighted_percentile.to_clipboard()

In [ ]:
category_map_exact.loc[repeated_schems_in_peers].to_clipboard()

In [ ]:
############################### for All scheme - SEBI Category based ###################################

q1_aum_per = category_map[['fund house']].merge(((qy_df_score == 'q1')*percent_aum_df1)[qy_df_score.columns].T,left_index=True,right_index=True,how='right')
q2_aum_per = category_map[['fund house']].merge(((qy_df_score == 'q2')*percent_aum_df1)[qy_df_score.columns].T,left_index=True,right_index=True,how='right')
q3_aum_per = category_map[['fund house']].merge(((qy_df_score == 'q3')*percent_aum_df1)[qy_df_score.columns].T,left_index=True,right_index=True,how='right')
q4_aum_per = category_map[['fund house']].merge(((qy_df_score == 'q4')*percent_aum_df1)[qy_df_score.columns].T,left_index=True,right_index=True,how='right')

percent_aum_in_q1 = q1_aum_per.groupby('fund house').sum()
percent_aum_in_q2 = q2_aum_per.groupby('fund house').sum()
percent_aum_in_q3 = q3_aum_per.groupby('fund house').sum()
percent_aum_in_q4 = q4_aum_per.groupby('fund house').sum()

no_of_schemes = category_map[['fund house']].merge((pd.isna(qy_df_score) == False).T,left_index=True,right_index=True,how='right').groupby('fund house').sum()

schemes_in_q1 = category_map[['fund house']].merge((qy_df_score == 'q1')[qy_df_score.columns].T,left_index=True,right_index=True,how='right').groupby('fund house').sum()
schemes_in_q2 = category_map[['fund house']].merge((qy_df_score == 'q2')[qy_df_score.columns].T,left_index=True,right_index=True,how='right').groupby('fund house').sum()
schemes_in_q3 = category_map[['fund house']].merge((qy_df_score == 'q3')[qy_df_score.columns].T,left_index=True,right_index=True,how='right').groupby('fund house').sum()
schemes_in_q4 = category_map[['fund house']].merge((qy_df_score == 'q4')[qy_df_score.columns].T,left_index=True,right_index=True,how='right').groupby('fund house').sum()

percent_schemes_in_q1 = schemes_in_q1/no_of_schemes
percent_schemes_in_q2 = schemes_in_q2/no_of_schemes
percent_schemes_in_q3 = schemes_in_q3/no_of_schemes
percent_schemes_in_q4 = schemes_in_q4/no_of_schemes

In [ ]:
#### Exporting to calculate in excel
export1 = category_map[['fund house']].merge(qy_df_score.T,left_index=True,right_index=True,how='right')
export1.index.names = ['Scheme']
export1 = export1.sort_values(by = ['fund house','Scheme'])

export2 = category_map[['fund house']].merge(percent_aum_df1.T,left_index=True,right_index=True,how='right')
export2 = export2.loc[export1.index]
export2.index.names = ['Scheme']
export2 = export2.sort_values(by = ['fund house','Scheme'])

# export1.to_clipboard()
# export2.to_clipboard()

In [ ]:
# Average score during the financial year  - 1Y, 2Y, 3Y all combined from score_df
### For Scheme -  what is the mean score
year_list = list(set((data.index).year))
year_list.sort()

year_list1 = year_list.copy()
year_list1.append(2025)

mean_score_df = pd.DataFrame()
for y in year_list1[4:] : 
    y_ = year_list1[year_list1.index(y)-1]
    end_date = pd.to_datetime(f'{y}-03-31')
    start_date = pd.to_datetime(f'{y_}-03-31')   
    mean_score_df.loc[y,score_df.columns] = score_df.loc[start_date:end_date].mean()
mean_score_df1 = mean_score_df.T.merge(category_map[['Category']],left_index=True,right_index=True,how='left').T
cats1 = ['Flexi Cap Fund','Small cap Fund','Mid Cap Fund','Large Cap Fund']
cats2 = cats1 + [i for i in categories if i not in cats1]
mean_score_df1.columns = pd.MultiIndex.from_tuples(list(zip(mean_score_df1.loc['Category'],mean_score_df1.columns)), names=["Category", "Scheme"])
mean_score_df1 = mean_score_df1.drop('Category',axis=0)
mean_score_df1 = mean_score_df1[cats2]

In [ ]:
mean_score_df1.to_clipboard()

In [ ]:
# What is the AMC wise AUM weighted score on daily basis and Fiscal year basis
applicable_schemes = list(category_map.loc[category_map['Category'].isin(categories)].index)
aum1 = aum.loc[applicable_schemes]

#  ffill in AUM since not all schemes have reported AUM historically on same periodicity
aum1 = aum1.T
aum1 = aum1.sort_index()
aum1 = aum1.replace('--',np.nan)
aum1 = aum1.ffill()
aum1 = aum1.T

for i in aum1.index : 
    aum1.loc[i,'fund house'] = i.split()[0]
aum1.insert(0,'fund house' ,aum1.pop('fund house')) 
#######################################################################################################################
# # LOB
fund_aum = aum1.groupby('fund house').sum()
funds = list(fund_aum.index)
amc_score_rolling_df = pd.DataFrame()
# for fund in funds : 
for fund in top15 :
    percent_aum = aum1.loc[aum1['fund house'].isin([fund])].iloc[:,1:]/fund_aum.loc[fund]
    percent_aum = percent_aum.T
    
    # making last availabale %of AUM (monthly AUM) as daily % of AUM 
    percent_aum_ = pd.DataFrame([1]*len(score_df) ,index = score_df.index)
    
    percent_aum_ = percent_aum_.merge(percent_aum,left_index=True,right_index=True,how='outer').ffill().drop(0,axis=1)
    percent_aum_ = percent_aum_.loc[score_df.index]

#     # AUM weighted rolling %tile of funds aggregated over AMC level
    amc_score_rolling = pd.DataFrame((score_df[percent_aum_.columns]*percent_aum_).sum(axis=1),columns = [fund])
    amc_score_rolling_df = amc_score_rolling_df.merge(amc_score_rolling,left_index=True,right_index=True,how='outer')
    
year_list = list(set((data.index).year))
year_list.sort()

year_list1 = year_list.copy()
year_list1.append(2025)

mean_score_df_amc = pd.DataFrame()
for y in year_list1[4:] : 
    y_ = year_list1[year_list1.index(y)-1]
    end_date = pd.to_datetime(f'{y}-03-31')
    start_date = pd.to_datetime(f'{y_}-03-31')   
    mean_score_df_amc.loc[y,amc_score_rolling_df.columns] = amc_score_rolling_df.loc[start_date:end_date].mean()

In [ ]:
amc_score_rolling_df.to_clipboard()
mean_score_df_amc.to_clipboard()

In [ ]:
################################ Plotting Etc  Below###########################################################################

In [ ]:
################# temp : What % of days in last 250 days, the scheme outperformed NSE 500 TRI ################################
bench = pd.read_csv("NSE500 TR till Feb11, 2025.csv")
bench.index = pd.to_datetime(bench.pop("Date"),format = '%d-%m-%Y')

birla_schemes = (category_map.loc[category_map['fund house']=='Aditya'].index)
data1 = data[birla_schemes]
common = bench.index.intersection(data1.index)
data1 = data1.loc[common]
bench = bench.loc[common]

birla_1y_rets = data1/data1.shift(250)-1
bench_1y_rets = bench/bench.shift(250)-1
birla_1y_alpha = birla_1y_rets.sub(bench_1y_rets['NIFTY 500'],axis=0)
birla_1y_alpha = birla_1y_alpha[pd.isna(birla_1y_alpha).sum(axis=1) != len(birla_1y_alpha.columns)]
############################################################################################################################

In [ ]:
birla_1y_alpha.to_clipboard()

In [ ]:
amc_q1q2_df.to_clipboard()

In [ ]:
visual_df

In [ ]:
visual_df

In [ ]:
birla_df_q4_percent.T.to_clipboard()

In [ ]:
aum_distri1.loc[aum_distri1.index.get_level_values('fund house').isin(['Aditya'])].T.to_clipboard()

In [ ]:
visual_df_q1q2.T.to_clipboard()
visual_df_q1q2_percent.T.to_clipboard()
aum_distri1.T.to_clipboard()
visual_df_q1q2_percent_rank.T.to_clipboard()

In [ ]:
########### % in Q1  +Q2 ################# HD

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio
from tqdm import tqdm
import matplotlib.colors as mcolors
import colorsys
from matplotlib.patches import Patch

# Optional: set a default higher DPI for all figures
plt.rcParams['figure.dpi'] = 150

# =============================================================================
# 1. DATA PREPROCESSING
# =============================================================================
# Assuming visual_df is your original DataFrame
df_reset = visual_df.reset_index()
df_long = df_reset.melt(
    id_vars=['Rolling Window', 'fund house', 'Breakdown', 'Quartile'],
    var_name='Date',
    value_name='AUM_pct'
)

rolling_window_choice = '6 months'
df_filtered = df_long[df_long['Rolling Window'] == rolling_window_choice].copy()

# Convert Date column to datetime (if not already)
df_filtered['Date'] = pd.to_datetime(df_filtered['Date'])

# Choose a fund house for the example (or loop over fund houses)
fund = df_filtered['fund house'].unique()[0]
fund_df = df_filtered[df_filtered['fund house'] == fund].copy()

# --- Grouping and Aggregation ---
agg_list = []

# Adjust as needed for your data
aum_distri = (category_map.loc[category_map['fund house'] == fund]
              .merge(percent_aum_df1.iloc[:, 1:], left_index=True, right_index=True, how='left')
              .groupby('Breakdown').sum().T)
aum_distri.index = pd.to_datetime(aum_distri.index)

for breakdown in sorted(fund_df['Breakdown'].unique()):
    sub_df = fund_df[fund_df['Breakdown'] == breakdown].copy()
    pivot_df = sub_df.pivot(index='Date', columns='Quartile', values='AUM_pct').sort_index()

    if '% in Q1' in pivot_df.columns and '% in Q2' in pivot_df.columns:
        pivot_df['Combined'] = pivot_df['% in Q1'] + pivot_df['% in Q2']
        pivot_df['% of total AMC AUM'] = aum_distri[breakdown][pivot_df.index]
        
        # Resample monthly (mean); adjust as needed (.sum(), .last(), etc.)
        monthly = pivot_df[['Combined', '% of total AMC AUM']].resample('M').mean()
        monthly = monthly.reset_index()
        monthly['Breakdown'] = breakdown
        
        agg_list.append(monthly)

if agg_list:
    agg_df = pd.concat(agg_list)
else:
    print("No data available with both '% in Q1' and '% in Q2'.")
    agg_df = pd.DataFrame()

# =============================================================================
# 2. PIVOT THE DATA
# =============================================================================
breakdowns = ["Asset Allocation", "Top Down", "Bottoms Up", "Thematic", "Arbitrage+"]

grouped_df_part = (agg_df.reset_index()
                          .pivot(index="Date", columns="Breakdown", values="Combined") * 100).sort_index()
grouped_df_total = (agg_df.reset_index()
                           .pivot(index="Date", columns="Breakdown", values="% of total AMC AUM") * 100).sort_index()

# =============================================================================
# 3. GENERATE A FRAME FOR EVERY DATE (MONTH) WITH MATPLOTLIB
# =============================================================================
frames_dir = "frames_monthly_matplotlib"
if not os.path.exists(frames_dir):
    os.makedirs(frames_dir)

base_colors = plt.cm.tab10.colors
n_breakdowns = len(breakdowns)
color_list = [base_colors[i % len(base_colors)] for i in range(n_breakdowns)]

group_width = 0.8
bar_width = group_width / n_breakdowns

unique_dates = grouped_df_total.index
start_date = pd.to_datetime('2023-01-01')
end_date = pd.to_datetime('2024-12-31')
unique_dates = [i for i in unique_dates if i >= start_date and i <= end_date]

frame_files = []

print("Generating frames for every Date...")
for date_val in tqdm(unique_dates, desc="Dates"):
    if (date_val not in grouped_df_total.index) or (date_val not in grouped_df_part.index):
        continue

    total_row = grouped_df_total.loc[date_val]
    part_row  = grouped_df_part.loc[date_val]

    # Increase figure size for better clarity
    fig, ax = plt.subplots(figsize=(10, 6))  # Adjust as needed (width, height)

    # We create a bar chart with ONE "group" on the x-axis (x=0).
    x_groups = np.array([0])

    for j, bd in enumerate(breakdowns):
        positions = x_groups - group_width/2 + (j + 0.5) * bar_width

        total_val = total_row.get(bd, 0)
        part_val  = part_row.get(bd, 0)
        base_color = color_list[j]

        # Plot the Total AUM bar
        ax.bar(
            positions,
            total_val,
            width=bar_width,
            color=base_color,
            edgecolor="black",
            label=f"{bd} Total % AUM"
        )

        # Plot the Q1+Q2 bar with a hatch pattern
        ax.bar(
            positions,
            part_val,
            width=bar_width * 0.6,
            color=base_color,
            edgecolor="black",
            hatch="///",
            label=f"{bd} % AUM in Q1+Q2"
        )

    ax.set_xticks([0])
    ax.set_xticklabels([date_val.strftime("%b %Y")])

    ax.set_xlabel("Date")
    ax.set_ylabel("% AUM")
    ax.set_title(f"Fund House: {fund} - Total AUM and % in Q1+Q2 ({date_val.strftime('%Y-%m')})")

    # Place the legend inside the plot at the top-right corner
    ax.legend(
        loc="upper right",   # Can be changed to "upper left", "upper center", etc.
        fontsize="xx-small", # Smaller legend font
        ncol=1               # Single column
    )

    ax.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()

    frame_filename = os.path.join(frames_dir, f"frame_{date_val.strftime('%Y%m')}.png")
    # Save at higher DPI for better resolution
    plt.savefig(frame_filename, bbox_inches='tight', dpi=200)
    plt.close(fig)
    frame_files.append(frame_filename)

# =============================================================================
# 4. CREATE THE ANIMATED GIF
# =============================================================================
if not frame_files:
    raise RuntimeError("No frames were generated successfully.")

print("Combining frames into GIF...")
frame_files = sorted(frame_files)
frames = [imageio.imread(filename) for filename in frame_files]

gif_filename = f'{fund} % in Q1+Q2.gif'
# Duration sets how long each frame displays, e.g., 1 second
imageio.mimsave(gif_filename, frames, duration=1)
print("Animation GIF saved as", gif_filename)


In [ ]:
########### % in Q4  ################# HD

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio
from tqdm import tqdm

# Optional: set a default higher DPI for all figures
plt.rcParams['figure.dpi'] = 150

# =============================================================================
# 1. DATA PREPROCESSING
# =============================================================================
# Assuming visual_df is your original DataFrame
df_reset = visual_df.reset_index()
df_long = df_reset.melt(
    id_vars=['Rolling Window', 'fund house', 'Breakdown', 'Quartile'],
    var_name='Date',
    value_name='AUM_pct'
)

rolling_window_choice = '6 months'
df_filtered = df_long[df_long['Rolling Window'] == rolling_window_choice].copy()

# Convert Date column to datetime (if not already)
df_filtered['Date'] = pd.to_datetime(df_filtered['Date'])

# Choose a fund house for the example (or loop over fund houses)
fund = df_filtered['fund house'].unique()[0]
fund_df = df_filtered[df_filtered['fund house'] == fund].copy()

# --- Grouping and Aggregation ---
agg_list = []

# Adjust as needed for your data
aum_distri = (category_map.loc[category_map['fund house'] == fund]
              .merge(percent_aum_df1.iloc[:, 1:], left_index=True, right_index=True, how='left')
              .groupby('Breakdown').sum().T)
aum_distri.index = pd.to_datetime(aum_distri.index)

for breakdown in sorted(fund_df['Breakdown'].unique()):
    sub_df = fund_df[fund_df['Breakdown'] == breakdown].copy()
    pivot_df = sub_df.pivot(index='Date', columns='Quartile', values='AUM_pct').sort_index()

    if '% in Q4' in pivot_df.columns : 
        pivot_df['Combined'] = pivot_df['% in Q4']
        pivot_df['% of total AMC AUM'] = aum_distri[breakdown][pivot_df.index]
        
        # Resample monthly (mean); adjust as needed (.sum(), .last(), etc.)
        monthly = pivot_df[['Combined', '% of total AMC AUM']].resample('M').mean()
        monthly = monthly.reset_index()
        monthly['Breakdown'] = breakdown
        
        agg_list.append(monthly)

if agg_list:
    agg_df = pd.concat(agg_list)
else:
    print("No data available with both '% in Q1' and '% in Q2'.")
    agg_df = pd.DataFrame()

# =============================================================================
# 2. PIVOT THE DATA
# =============================================================================
breakdowns = ["Asset Allocation", "Top Down", "Bottoms Up", "Thematic", "Arbitrage+"]

grouped_df_part = (agg_df.reset_index()
                          .pivot(index="Date", columns="Breakdown", values="Combined") * 100).sort_index()
grouped_df_total = (agg_df.reset_index()
                           .pivot(index="Date", columns="Breakdown", values="% of total AMC AUM") * 100).sort_index()

# =============================================================================
# 3. GENERATE A FRAME FOR EVERY DATE (MONTH) WITH MATPLOTLIB
# =============================================================================
frames_dir = "frames_monthly_matplotlib"
if not os.path.exists(frames_dir):
    os.makedirs(frames_dir)

base_colors = plt.cm.tab10.colors
n_breakdowns = len(breakdowns)
color_list = [base_colors[i % len(base_colors)] for i in range(n_breakdowns)]

group_width = 0.8
bar_width = group_width / n_breakdowns

unique_dates = grouped_df_total.index
start_date = pd.to_datetime('2022-01-01')
unique_dates = [i for i in unique_dates if i >= start_date]

frame_files = []

print("Generating frames for every Date...")
for date_val in tqdm(unique_dates, desc="Dates"):
    if (date_val not in grouped_df_total.index) or (date_val not in grouped_df_part.index):
        continue

    total_row = grouped_df_total.loc[date_val]
    part_row  = grouped_df_part.loc[date_val]

    # Increase figure size for better clarity
    fig, ax = plt.subplots(figsize=(10, 6))  # Adjust as needed (width, height)

    # We create a bar chart with ONE "group" on the x-axis (x=0).
    x_groups = np.array([0])

    for j, bd in enumerate(breakdowns):
        positions = x_groups - group_width/2 + (j + 0.5) * bar_width

        total_val = total_row.get(bd, 0)
        part_val  = part_row.get(bd, 0)
        base_color = color_list[j]

        # Plot the Total AUM bar
        ax.bar(
            positions,
            total_val,
            width=bar_width,
            color=base_color,
            edgecolor="black",
            label=f"{bd} Total % AUM"
        )

        # Plot the Q1+Q2 bar with a hatch pattern
        ax.bar(
            positions,
            part_val,
            width=bar_width * 0.6,
            color=base_color,
            edgecolor="black",
            hatch="///",
            label=f"{bd} % AUM in Q4"
        )

    ax.set_xticks([0])
    ax.set_xticklabels([date_val.strftime("%b %Y")])

    ax.set_xlabel("Date")
    ax.set_ylabel("% AUM")
    ax.set_title(f"Fund House: {fund} - Total AUM and % in Q4 ({date_val.strftime('%Y-%m')})")

    # Place the legend inside the plot at the top-right corner
    ax.legend(
        loc="upper right",   # Can be changed to "upper left", "upper center", etc.
        fontsize="xx-small", # Smaller legend font
        ncol=1               # Single column
    )

    ax.grid(axis="y", linestyle="--", alpha=0.7)
    plt.tight_layout()

    frame_filename = os.path.join(frames_dir, f"frame_{date_val.strftime('%Y%m')}.png")
    # Save at higher DPI for better resolution
    plt.savefig(frame_filename, bbox_inches='tight', dpi=200)
    plt.close(fig)
    frame_files.append(frame_filename)

# =============================================================================
# 4. CREATE THE ANIMATED GIF
# =============================================================================
if not frame_files:
    raise RuntimeError("No frames were generated successfully.")

print("Combining frames into GIF...")
frame_files = sorted(frame_files)
frames = [imageio.imread(filename) for filename in frame_files]

gif_filename = f'{fund} % in Q4.gif'
# Duration sets how long each frame displays, e.g., 1 second
imageio.mimsave(gif_filename, frames, duration=1)
print("Animation GIF saved as", gif_filename)


In [ ]:
# ### % in Q1  +Q2 - Bad picture Quality
# # --- Data Preprocessing ---
# # Assuming visual_df is your original DataFrame
# df_reset = visual_df.reset_index()
# df_long = df_reset.melt(
#     id_vars=['Rolling Window', 'fund house', 'Breakdown', 'Quartile'],
#     var_name='Date',
#     value_name='AUM_pct'
# )

# # rolling_window_choice = '1 Year'
# rolling_window_choice = '6 months'
# df_filtered = df_long[df_long['Rolling Window'] == rolling_window_choice].copy()

# # Convert Date column to datetime (if not already)
# df_filtered['Date'] = pd.to_datetime(df_filtered['Date'])

# # Choose a fund house for the example (or loop over fund houses)
# fund = df_filtered['fund house'].unique()[0]
# fund_df = df_filtered[df_filtered['fund house'] == fund].copy()

# # --- Grouping and Aggregation ---
# # We will aggregate the data by a chosen time period (e.g., month)
# # and compute the combined % AUM in Q1+Q2 for each breakdown.

# # Create an empty list to store aggregated data
# agg_list = []
# aum_distri = category_map.loc[category_map['fund house']==fund].merge(percent_aum_df1.iloc[:,1:],left_index=True,right_index=True,how='left').groupby('Breakdown').sum().T
# aum_distri.index = pd.to_datetime(aum_distri.index)

# for breakdown in sorted(fund_df['Breakdown'].unique()):
#     sub_df = fund_df[fund_df['Breakdown'] == breakdown].copy()
    
#     # Pivot the data so that each row is a Date and columns are quartiles
#     pivot_df = sub_df.pivot(index='Date', columns='Quartile', values='AUM_pct')
#     pivot_df = pivot_df.sort_index()
    
#     # We need both '% in Q1' and '% in Q2' to compute the combined value.
#     if '% in Q1' in pivot_df.columns and '% in Q2' in pivot_df.columns:
#         # Create a new column with the combined value (converted to percentage)
#         pivot_df['Combined'] = (pivot_df['% in Q1'] + pivot_df['% in Q2'])
#         pivot_df['% of total AMC AUM'] = aum_distri[breakdown][pivot_df.index]
        
# #         ## Resample/Group the data by month (or any period you prefer)
# #         monthly = pivot_df['Combined'].resample('M').mean()  # you can also use .last(), .sum(), etc.
#         monthly = pivot_df[['Combined','% of total AMC AUM']].resample('M').mean()  # you can also use .last(), .sum(), etc.
        
#         monthly = monthly.reset_index()
#         monthly['Breakdown'] = breakdown
        
#         agg_list.append(monthly)
#     else:
#         # If the necessary columns aren't available, you might want to skip or set NaN.
#         continue

# # Concatenate all the aggregated data
# if agg_list:
#     agg_df = pd.concat(agg_list)
# else:
#     print("No data available with both '% in Q1' and '% in Q2'.")
#     agg_df = pd.DataFrame()
    
# import os
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import imageio
# from tqdm import tqdm

# breakdowns = ["Asset Allocation", "Top Down", "Bottoms Up", "Thematic","Arbitrage+"]
# # =============================================================================
# # 2. PIVOT THE DATA
# # =============================================================================
# # Pivot so each breakdown is a column. Multiply by 100 to convert fractions to percentages.
# grouped_df_part = (agg_df.reset_index().pivot(index="Date", columns="Breakdown", values="Combined") * 100).sort_index()
# grouped_df_total = (agg_df.reset_index().pivot(index="Date", columns="Breakdown", values="% of total AMC AUM") * 100).sort_index()

# # =============================================================================
# # 3. GENERATE A FRAME FOR EVERY DATE (MONTH) WITH MATPLOTLIB
# # =============================================================================
# # We will now iterate over *each* monthly index (instead of each year).
# frames_dir = "frames_monthly_matplotlib"
# if not os.path.exists(frames_dir):
#     os.makedirs(frames_dir)

# # Use Matplotlib’s built-in colormap (we use 'tab10' for distinct colors)
# base_colors = plt.cm.tab10.colors  # returns a tuple of RGBA values
# n_breakdowns = len(breakdowns)
# color_list = [base_colors[i % len(base_colors)] for i in range(n_breakdowns)]

# # Bar positioning parameters:
# group_width = 0.8
# bar_width = group_width / n_breakdowns

# # (Optional) Specify a fund name for the title:
# # fund = "YourFundName"

# unique_dates = grouped_df_total.index
# start_date = pd.to_datetime('2022-01-01')
# unique_dates = [i for i in unique_dates if i >= start_date]

# frame_files = []

# print("Generating frames for every Date...")
# for date_val in tqdm(unique_dates, desc="Dates"):
#     # Get a single row for this specific date
#     # (If your real data might have missing rows, handle it accordingly.)
#     if date_val not in grouped_df_total.index or date_val not in grouped_df_part.index:
#         continue

#     # For a single date, we have exactly 1 row in each DataFrame
#     total_row = grouped_df_total.loc[date_val]  # Series of length = number of breakdowns
#     part_row  = grouped_df_part.loc[date_val]   # Same shape

#     # We will create a bar chart with ONE "group" on the x-axis (x=0),
#     # and within that group, we'll place one pair of bars per breakdown:
#     x_groups = np.array([0])  # only one group
#     fig, ax = plt.subplots(figsize=(6, 4))

#     for j, bd in enumerate(breakdowns):
#         # positions for this breakdown within the single group
#         positions = x_groups - group_width/2 + (j + 0.5) * bar_width

#         total_val = total_row[bd]  # single float
#         part_val  = part_row[bd]   # single float
#         base_color = color_list[j]

#         # Plot the Total AUM bar (full-width, solid color)
#         ax.bar(positions,
#                total_val,
#                width=bar_width,
#                color=base_color,
#                edgecolor="black",
# #                label=f"{bd} Total AUM" if j == 0 else "",
#                label=f"{bd} Total AUM" )

#         # Plot the Q1+Q2 bar (narrower, with a hatch pattern)
#         ax.bar(positions,
#                part_val,
#                width=bar_width * 0.6,
#                color=base_color,
#                edgecolor="black",
#                hatch="///",
# #                label=f"{bd} Q1+Q2" if j == 0 else "",
#                label=f"{bd} AUM in Q1+Q2" )

#     # For a single group, we place the tick at 0
#     ax.set_xticks([0])
#     # Show the month-year label for clarity
#     ax.set_xticklabels([date_val.strftime("%b %Y")])

#     ax.set_xlabel("Date")
#     ax.set_ylabel("% AUM")
#     ax.set_title(f"Fund House: {fund} - Total AUM and Q1+Q2 ({date_val.strftime('%Y-%m')})")
#     ax.legend(loc="upper right", fontsize="small", ncol=2)
#     ax.grid(axis="y", linestyle="--", alpha=0.7)

#     plt.tight_layout()
#     frame_filename = os.path.join(frames_dir, f"frame_{date_val.strftime('%Y%m')}.png")
#     plt.savefig(frame_filename)
#     plt.close(fig)
#     frame_files.append(frame_filename)

# # =============================================================================
# # 4. CREATE THE ANIMATED GIF
# # =============================================================================
# if not frame_files:
#     raise RuntimeError("No frames were generated successfully.")

# print("Combining frames into GIF...")
# # Sort frame files by date label if needed
# frame_files = sorted(frame_files)
# frames = [imageio.imread(filename) for filename in frame_files]
# gif_filename = f'{fund} % in Q1+Q2.gif'
# imageio.mimsave(gif_filename, frames, duration=1)  # 1 second per frame
# print("Animation GIF saved as", gif_filename)


In [ ]:
df_filtered['fund house']

In [ ]:
# ### % in Q4 - good picture Quality

# # --- Data Preprocessing ---
# # Assuming visual_df is your original DataFrame
# df_reset = visual_df.reset_index()
# df_long = df_reset.melt(
#     id_vars=['Rolling Window', 'fund house', 'Breakdown', 'Quartile'],
#     var_name='Date',
#     value_name='AUM_pct'
# )

# rolling_window_choice = '1 Year'
# rolling_window_choice = '6 months'

# df_filtered = df_long[df_long['Rolling Window'] == rolling_window_choice].copy()

# # Convert Date column to datetime (if not already)
# df_filtered['Date'] = pd.to_datetime(df_filtered['Date'])

# # Choose a fund house for the example (or loop over fund houses)
# fund = df_filtered['fund house'].unique()[0]
# fund_df = df_filtered[df_filtered['fund house'] == fund].copy()

# # --- Grouping and Aggregation ---
# # We will aggregate the data by a chosen time period (e.g., month)
# # and compute the combined % AUM in Q1+Q2 for each breakdown.

# # Create an empty list to store aggregated data
# agg_list = []
# aum_distri = category_map.loc[category_map['fund house']==fund].merge(percent_aum_df1.iloc[:,1:],left_index=True,right_index=True,how='left').groupby('Breakdown').sum().T
# aum_distri.index = pd.to_datetime(aum_distri.index)

# for breakdown in sorted(fund_df['Breakdown'].unique()):
#     sub_df = fund_df[fund_df['Breakdown'] == breakdown].copy()
    
#     # Pivot the data so that each row is a Date and columns are quartiles
#     pivot_df = sub_df.pivot(index='Date', columns='Quartile', values='AUM_pct')
#     pivot_df = pivot_df.sort_index()
    
#     # We need both '% in Q1' and '% in Q2' to compute the combined value.
#     if '% in Q1' in pivot_df.columns and '% in Q2' in pivot_df.columns:
#         # Create a new column with the combined value (converted to percentage)
#         pivot_df['Combined'] = (pivot_df['% in Q4'])
#         pivot_df['% of total AMC AUM'] = aum_distri[breakdown][pivot_df.index]
        
# #         ## Resample/Group the data by month (or any period you prefer)
# #         monthly = pivot_df['Combined'].resample('M').mean()  # you can also use .last(), .sum(), etc.
#         monthly = pivot_df[['Combined','% of total AMC AUM']].resample('M').mean()  # you can also use .last(), .sum(), etc.
        
#         monthly = monthly.reset_index()
#         monthly['Breakdown'] = breakdown
        
#         agg_list.append(monthly)
#     else:
#         # If the necessary columns aren't available, you might want to skip or set NaN.
#         continue

# # Concatenate all the aggregated data
# if agg_list:
#     agg_df = pd.concat(agg_list)
# else:
#     print("No data available with both '% in Q1' and '% in Q2'.")
#     agg_df = pd.DataFrame()

# import os
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import imageio
# from tqdm import tqdm

# breakdowns = ["Asset Allocation", "Top Down", "Bottoms Up", "Thematic","Arbitrage+"]
# # =============================================================================
# # 2. PIVOT THE DATA
# # =============================================================================
# # Pivot so each breakdown is a column. Multiply by 100 to convert fractions to percentages.
# grouped_df_part = (agg_df.reset_index().pivot(index="Date", columns="Breakdown", values="Combined") * 100).sort_index()
# grouped_df_total = (agg_df.reset_index().pivot(index="Date", columns="Breakdown", values="% of total AMC AUM") * 100).sort_index()

# # =============================================================================
# # 3. GENERATE A FRAME FOR EVERY DATE (MONTH) WITH MATPLOTLIB
# # =============================================================================
# # We will now iterate over *each* monthly index (instead of each year).
# frames_dir = "frames_monthly_matplotlib"
# if not os.path.exists(frames_dir):
#     os.makedirs(frames_dir)

# # Use Matplotlib’s built-in colormap (we use 'tab10' for distinct colors)
# base_colors = plt.cm.tab10.colors  # returns a tuple of RGBA values
# n_breakdowns = len(breakdowns)
# color_list = [base_colors[i % len(base_colors)] for i in range(n_breakdowns)]

# # Bar positioning parameters:
# group_width = 0.8
# bar_width = group_width / n_breakdowns

# # (Optional) Specify a fund name for the title:
# # fund = "YourFundName"

# unique_dates = grouped_df_total.index
# start_date = pd.to_datetime('2022-01-01')
# unique_dates = [i for i in unique_dates if i >= start_date]

# frame_files = []

# print("Generating frames for every Date...")
# for date_val in tqdm(unique_dates, desc="Dates"):
#     # Get a single row for this specific date
#     # (If your real data might have missing rows, handle it accordingly.)
#     if date_val not in grouped_df_total.index or date_val not in grouped_df_part.index:
#         continue

#     # For a single date, we have exactly 1 row in each DataFrame
#     total_row = grouped_df_total.loc[date_val]  # Series of length = number of breakdowns
#     part_row  = grouped_df_part.loc[date_val]   # Same shape

#     # We will create a bar chart with ONE "group" on the x-axis (x=0),
#     # and within that group, we'll place one pair of bars per breakdown:
#     x_groups = np.array([0])  # only one group
#     fig, ax = plt.subplots(figsize=(6, 4))

#     for j, bd in enumerate(breakdowns):
#         # positions for this breakdown within the single group
#         positions = x_groups - group_width/2 + (j + 0.5) * bar_width

#         total_val = total_row[bd]  # single float
#         part_val  = part_row[bd]   # single float
#         base_color = color_list[j]

#         # Plot the Total AUM bar (full-width, solid color)
#         ax.bar(positions,
#                total_val,
#                width=bar_width,
#                color=base_color,
#                edgecolor="black",
# #                label=f"{bd} Total AUM" if j == 0 else "",
#                label=f"{bd} Total AUM" )

#         # Plot the Q1+Q2 bar (narrower, with a hatch pattern)
#         ax.bar(positions,
#                part_val,
#                width=bar_width * 0.6,
#                color=base_color,
#                edgecolor="black",
#                hatch="///",
# #                label=f"{bd} Q1+Q2" if j == 0 else "",
#                label=f"{bd}  AUM in Q4"" )

#     # For a single group, we place the tick at 0
#     ax.set_xticks([0])
#     # Show the month-year label for clarity
#     ax.set_xticklabels([date_val.strftime("%b %Y")])

#     ax.set_xlabel("Date")
#     ax.set_ylabel("% AUM")
#     ax.set_title(f"Fund House: {fund} - Total AUM and AUM in Q4 ({date_val.strftime('%Y-%m')})")
#     ax.legend(loc="upper right", fontsize="small", ncol=2)
#     ax.grid(axis="y", linestyle="--", alpha=0.7)

#     plt.tight_layout()
#     frame_filename = os.path.join(frames_dir, f"frame_{date_val.strftime('%Y%m')}.png")
#     plt.savefig(frame_filename)
#     plt.close(fig)
#     frame_files.append(frame_filename)

# # =============================================================================
# # 4. CREATE THE ANIMATED GIF
# # =============================================================================
# if not frame_files:
#     raise RuntimeError("No frames were generated successfully.")

# print("Combining frames into GIF...")
# # Sort frame files by date label if needed
# frame_files = sorted(frame_files)
# frames = [imageio.imread(filename) for filename in frame_files]
# gif_filename = f'{fund} % in Q4.gif'
# imageio.mimsave(gif_filename, frames, duration=1/3)  # 1 second per frame
# print("Animation GIF saved as", gif_filename)


In [ ]:
################################################################################################################################